In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs (they are in current working directory)
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_error = pd.read_csv('error_logs.csv')

# Helper to produce compact top counts string
def top_counts_series(series, top_n=20):
    s = series.value_counts().head(top_n)
    return ", ".join([f"{idx}({cnt})" for idx, cnt in s.items()])

def ts_to_local_str(ts_seconds):
    if pd.isna(ts_seconds):
        return None
    # convert scalar seconds to localized string in Asia/Shanghai (UTC+8)
    return pd.to_datetime(int(ts_seconds), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %z')

# Summaries for each file
# Metric
metric_total_rows = len(df_metric)
metric_min_ts = df_metric['timestamp'].min() if metric_total_rows>0 else None
metric_max_ts = df_metric['timestamp'].max() if metric_total_rows>0 else None
metric_unique_cmdb = df_metric['cmdb_id'].nunique()
metric_top_cmdb = top_counts_series(df_metric['cmdb_id'], top_n=20)
metric_top_kpi = top_counts_series(df_metric['kpi_name'], top_n=20)

# Trace
trace_total_rows = len(df_trace)
trace_min_ts = df_trace['timestamp'].min() if trace_total_rows>0 else None
trace_max_ts = df_trace['timestamp'].max() if trace_total_rows>0 else None
trace_unique_cmdb = df_trace['cmdb_id'].nunique()
trace_top_cmdb = top_counts_series(df_trace['cmdb_id'], top_n=20)
trace_top_name = top_counts_series(df_trace['trace_name'], top_n=20)

# Log
log_total_rows = len(df_log)
log_min_ts = df_log['timestamp'].min() if log_total_rows>0 else None
log_max_ts = df_log['timestamp'].max() if log_total_rows>0 else None
log_unique_cmdb = df_log['cmdb_id'].nunique()
log_top_cmdb = top_counts_series(df_log['cmdb_id'], top_n=20)
log_top_name = top_counts_series(df_log['log_name'], top_n=20)

# Error logs (top 10 messages)
err_total_rows = len(df_error)
err_min_ts = df_error['timestamp'].min() if err_total_rows>0 else None
err_max_ts = df_error['timestamp'].max() if err_total_rows>0 else None
err_unique_cmdb = df_error['cmdb_id'].nunique()
err_top_cmdb = top_counts_series(df_error['cmdb_id'], top_n=20)
# top 10 distinct messages with counts
err_top_messages = top_counts_series(df_error['message'], top_n=10)

# Assemble summary DataFrame (one row per file)
summary = pd.DataFrame([
    {
        'file': 'metric.csv',
        'total_rows': metric_total_rows,
        'min_timestamp(Asia/Shanghai)': ts_to_local_str(metric_min_ts),
        'max_timestamp(Asia/Shanghai)': ts_to_local_str(metric_max_ts),
        'unique_cmdb_id_count': metric_unique_cmdb,
        'top_20_cmdb_id_with_counts': metric_top_cmdb,
        'unique_kpi_name_with_counts_top20': metric_top_kpi
    },
    {
        'file': 'trace.csv',
        'total_rows': trace_total_rows,
        'min_timestamp(Asia/Shanghai)': ts_to_local_str(trace_min_ts),
        'max_timestamp(Asia/Shanghai)': ts_to_local_str(trace_max_ts),
        'unique_cmdb_id_count': trace_unique_cmdb,
        'top_20_cmdb_id_with_counts': trace_top_cmdb,
        'unique_trace_name_with_counts_top20': trace_top_name
    },
    {
        'file': 'log.csv',
        'total_rows': log_total_rows,
        'min_timestamp(Asia/Shanghai)': ts_to_local_str(log_min_ts),
        'max_timestamp(Asia/Shanghai)': ts_to_local_str(log_max_ts),
        'unique_cmdb_id_count': log_unique_cmdb,
        'top_20_cmdb_id_with_counts': log_top_cmdb,
        'unique_log_name_with_counts_top20': log_top_name
    },
    {
        'file': 'error_logs.csv',
        'total_rows': err_total_rows,
        'min_timestamp(Asia/Shanghai)': ts_to_local_str(err_min_ts),
        'max_timestamp(Asia/Shanghai)': ts_to_local_str(err_max_ts),
        'unique_cmdb_id_count': err_unique_cmdb,
        'top_20_cmdb_id_with_counts': err_top_cmdb,
        'top_10_distinct_messages_with_counts': err_top_messages
    }
])

# Keep output compact: selected columns and up to 4 rows
summary[['file','total_rows','min_timestamp(Asia/Shanghai)','max_timestamp(Asia/Shanghai)',
         'unique_cmdb_id_count','top_20_cmdb_id_with_counts']].head(20), summary[['file','total_rows','min_timestamp(Asia/Shanghai)','max_timestamp(Asia/Shanghai)',
         'unique_cmdb_id_count','top_20_cmdb_id_with_counts','unique_kpi_name_with_counts_top20',
         'unique_trace_name_with_counts_top20','unique_log_name_with_counts_top20','top_10_distinct_messages_with_counts']].head(20)

```
Out[1]:
```
```python
summary_text = (
    "Summary of telemetry files:\n\n"
    "metric.csv:\n"
    "  total_rows: 19671\n"
    "  min_timestamp (UTC+8): 2021-03-23 18:30:00\n"
    "  max_timestamp (UTC+8): 2021-03-23 19:00:00\n"
    "  unique_cmdb_id_count: 25\n"
    "  top_20_cmdb_id_with_counts (truncated): Mysql02(3613), Mysql01(3605), Tomcat03(1764), ...\n"
    "  top_kpi_name_with_counts (top examples): app.sr(341), app.rr(341), app.mrt(341), app.cn...\n\n"
    "trace.csv:\n"
    "  total_rows: 7041\n"
    "  min_timestamp (UTC+8): 2021-03-23 18:30:00\n"
    "  max_timestamp (UTC+8): 2021-03-23 19:00:00\n"
    "  unique_cmdb_id_count: 12\n"
    "  top_20_cmdb_id_with_counts (truncated): MG01(1203), MG02(1197), Tomcat02(468), Tomcat0...\n"
    "  top_trace_name_with_counts (top examples): trace.self.duration_mean(372), trace.self.dura...\n\n"
    "log.csv:\n"
    "  total_rows: 486\n"
    "  min_timestamp (UTC+8): 2021-03-23 18:30:00\n"
    "  max_timestamp (UTC+8): 2021-03-23 19:00:00\n"
    "  unique_cmdb_id_count: 8\n"
    "  top_20_cmdb_id_with_counts (truncated): Tomcat04(76), Tomcat01(74), Tomcat03(74), Tomc...\n"
    "  top_log_name_with_counts (top examples): log.error_count(218), log.row_count(218), log....\n\n"
    "error_logs.csv:\n"
    "  total_rows: 0\n"
    "  min_timestamp (UTC+8): None\n"
    "  max_timestamp (UTC+8): None\n"
    "  unique_cmdb_id_count: 0\n"
    "  top_10_distinct_messages_with_counts: None\n\n"
    "Notes: Lists above are truncated for brevity; full top lists were returned by the earlier query."
)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(             file  total_rows min_timestamp(Asia/Shanghai) max_timestamp(Asia/Shanghai)  unique_cmdb_id_count                         top_20_cmdb_id_with_counts
0      metric.csv       19671    2021-03-23 18:30:00 +0800    2021-03-23 19:00:00 +0800                    25  Mysql02(3613), Mysql01(3605), Tomcat03(1764), ...
1       trace.csv        7041    2021-03-23 18:30:00 +0800    2021-03-23 19:00:00 +0800                    12  MG01(1203), MG02(1197), Tomcat02(468), Tomcat0...
2         log.csv         486    2021-03-23 18:30:00 +0800    2021-03-23 19:00:00 +0800                     8  Tomcat04(76), Tomcat01(74), Tomcat03(74), Tomc...
3  error_logs.csv           0                         None                         None                     0                                                   ,              file  total_rows min_timestamp(Asia/Shanghai) max_timestamp(Asia/Shanghai)  unique_cmdb_id_count                         top_20_cmdb_id_with_counts                  unique_kpi_name_with_counts_top20                unique_trace_name_with_counts_top20                  unique_log_name_with_counts_top20 top_10_distinct_messages_with_counts
0      metric.csv       19671    2021-03-23 18:30:00 +0800    2021-03-23 19:00:00 +0800                    25  Mysql02(3613), Mysql01(3605), Tomcat03(1764), ...  app.sr(341), app.rr(341), app.mrt(341), app.cn...                                                NaN                                                NaN                                  NaN
1       trace.csv        7041    2021-03-23 18:30:00 +0800    2021-03-23 19:00:00 +0800                    12  MG01(1203), MG02(1197), Tomcat02(468), Tomcat0...                                                NaN  trace.self.duration_mean(372), trace.self.dura...                                                NaN                                  NaN
2         log.csv         486    2021-03-23 18:30:00 +0800    2021-03-23 19:00:00 +0800                     8  Tomcat04(76), Tomcat01(74), Tomcat03(74), Tomc...                                                NaN                                                NaN  log.error_count(218), log.row_count(218), log....                                  NaN
3  error_logs.csv           0                         None                         None                     0                                                                                                   NaN                                                NaN                                                NaN                                     )```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# reuse df_metric if available, otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

tz = pytz.timezone('Asia/Shanghai')

# 1) Global aggregation by (cmdb_id, kpi_name) using the full series (rule 9)
grouped = df_metric.groupby(['cmdb_id', 'kpi_name'])

agg_global = grouped.agg(
    total_points=('value', 'count'),
    global_p95=('value', lambda s: float(s.quantile(0.95))),
    min_timestamp=('timestamp', 'min'),
    max_timestamp=('timestamp', 'max')
).reset_index()

# 2) Define incident window in UTC based on local Asia/Shanghai times
start_local = pd.Timestamp('2021-03-23 18:30:00', tz=tz)
end_local   = pd.Timestamp('2021-03-23 19:00:00', tz=tz)
# convert to UTC epoch seconds for filtering (timestamps are epoch seconds UTC)
start_epoch = int(start_local.tz_convert('UTC').timestamp())
end_epoch   = int(end_local.tz_convert('UTC').timestamp())

# 3) Filter metric rows to the incident window
df_incident = df_metric[(df_metric['timestamp'] >= start_epoch) & (df_metric['timestamp'] <= end_epoch)]

# overall total number of metric rows in incident window
incident_total_rows = int(df_incident.shape[0])

# 4) Aggregate incident window by group
if incident_total_rows > 0:
    agg_incident = df_incident.groupby(['cmdb_id', 'kpi_name']).agg(
        incident_points=('value', 'count'),
        incident_min_timestamp=('timestamp', 'min'),
        incident_max_timestamp=('timestamp', 'max')
    ).reset_index()
else:
    # empty frame with expected columns
    agg_incident = pd.DataFrame(columns=['cmdb_id','kpi_name','incident_points','incident_min_timestamp','incident_max_timestamp'])

# 5) Merge global and incident aggregates
merged = pd.merge(
    agg_global,
    agg_incident,
    on=['cmdb_id', 'kpi_name'],
    how='left'
)

# fill missing incident counts with 0
merged['incident_points'] = merged['incident_points'].fillna(0).astype(int)
# keep incident timestamps as NaN when no points

# 6) Convert epoch timestamps to Asia/Shanghai readable strings for display (compact)
def epoch_to_local_str(v):
    if pd.isna(v):
        return None
    return pd.to_datetime(int(v), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S')

for col in ['min_timestamp','max_timestamp','incident_min_timestamp','incident_max_timestamp']:
    merged[col] = merged[col].apply(lambda x: epoch_to_local_str(x) if pd.notna(x) else None)

# 7) Prepare final compact table sorted by incident_points desc, limit top 20
selected_cols = [
    'cmdb_id','kpi_name','total_points','global_p95',
    'min_timestamp','max_timestamp',
    'incident_points','incident_min_timestamp','incident_max_timestamp'
]
final_df = merged[selected_cols].sort_values(by='incident_points', ascending=False).head(20)

# Display compact results: the aggregated top-20 table and overall incident row count
final_df, incident_total_rows

```
Out[2]:
```
```python
# Prepare a concise plain-English summary string using the previously computed results
# Reuse variables final_df and incident_total_rows from the session.

# If final_df or incident_total_rows are not in scope, reload aggregates (fallback)
try:
    final_df
    incident_total_rows
except NameError:
    import pandas as pd, pytz
    df_metric = pd.read_csv('metric.csv')
    tz = pytz.timezone('Asia/Shanghai')
    # Minimal recompute (shouldn't happen in normal flow)
    grouped = df_metric.groupby(['cmdb_id', 'kpi_name']).agg(
        total_points=('value','count'),
        global_p95=('value', lambda s: float(s.quantile(0.95))),
        min_timestamp=('timestamp','min'),
        max_timestamp=('timestamp','max')
    ).reset_index()
    start_local = pd.Timestamp('2021-03-23 18:30:00', tz=tz)
    end_local   = pd.Timestamp('2021-03-23 19:00:00', tz=tz)
    start_epoch = int(start_local.tz_convert('UTC').timestamp())
    end_epoch   = int(end_local.tz_convert('UTC').timestamp())
    df_incident = df_metric[(df_metric['timestamp'] >= start_epoch) & (df_metric['timestamp'] <= end_epoch)]
    incident_total_rows = int(df_incident.shape[0])
    agg_incident = df_incident.groupby(['cmdb_id','kpi_name']).agg(
        incident_points=('value','count'),
        incident_min_timestamp=('timestamp','min'),
        incident_max_timestamp=('timestamp','max')
    ).reset_index() if incident_total_rows>0 else pd.DataFrame(columns=['cmdb_id','kpi_name','incident_points'])
    merged = pd.merge(grouped, agg_incident, on=['cmdb_id','kpi_name'], how='left')
    merged['incident_points'] = merged['incident_points'].fillna(0).astype(int)
    final_df = merged[['cmdb_id','kpi_name','total_points','global_p95','min_timestamp','max_timestamp','incident_points']].sort_values('incident_points', ascending=False).head(20)

# Build summary text
examples = []
for i, row in enumerate(final_df.head(5).itertuples(index=False), start=1):
    # row fields: cmdb_id, kpi_name, total_points, global_p95, min_timestamp, max_timestamp, incident_points
    examples.append(f"{i}) {row.cmdb_id} — {row.kpi_name} — total_points={int(row.total_points)} — global_p95={float(row.global_p95):,.3g}")

summary_text = (
    "Metric file incident-window summary (2021-03-23 18:30 to 19:00, UTC+8):\n"
    f"- Total metric rows in the incident window: {int(incident_total_rows)}\n"
    f"- Top 20 (by incident_points): all top groups shown have incident_points = 31 and incident time range 2021-03-23 18:30:00 to 2021-03-23 19:00:00 (UTC+8).\n"
    "- Example top groups:\n  " + "\n  ".join(examples) + "\n"
    "- Note: the table returned contains the requested columns for the top 20 groups (cmdb_id, kpi_name, total_points, global_p95, min_timestamp, max_timestamp, incident_points, incident_min_timestamp, incident_max_timestamp)."
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name  total_points    global_p95        min_timestamp        max_timestamp  incident_points incident_min_timestamp incident_max_timestamp
755   Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...            31  2.829756e+06  2021-03-23 18:30:00  2021-03-23 19:00:00               31    2021-03-23 18:30:00    2021-03-23 19:00:00
6         IG01           JVM-Threads_7778_JVM_ThreadCount_Threads            31  7.650000e+01  2021-03-23 18:30:00  2021-03-23 19:00:00               31    2021-03-23 18:30:00    2021-03-23 19:00:00
1113  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...            31  1.390000e+02  2021-03-23 18:30:00  2021-03-23 19:00:00               31    2021-03-23 18:30:00    2021-03-23 19:00:00
3         IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed            31  5.865392e+07  2021-03-23 18:30:00  2021-03-23 19:00:00               31    2021-03-23 18:30:00    2021-03-23 19:00:00
1114  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...            31  2.304870e+05  2021-03-23 18:30:00  2021-03-23 19:00:00               31    2021-03-23 18:30:00    2021-03-23 19:00:00
1131  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...            31  5.000000e+02  2021-03-23 18:30:00  2021-03-23 19:00:00               31    2021-03-23 18:30:00    2021-03-23 19:00:00
1126  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...            31  0.000000e+00  2021-03-23 18:30:00  2021-03-23 19:00:00               31    2021-03-23 18:30:00    2021-03-23 19:00:00
1127  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...            31  0.000000e+00  2021-03-23 18:30:00  2021-03-23 19:00:00               31    2021-03-23 18:30:00    2021-03-23 19:00:00
24        IG02           JVM-Threads_7778_JVM_ThreadCount_Threads            31  7.550000e+01  2021-03-23 18:30:00  2021-03-23 19:00:00               31    2021-03-23 18:30:00    2021-03-23 19:00:00
18        IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax            31  2.040136e+09  2021-03-23 18:30:00  2021-03-23 19:00:00               31    2021-03-23 18:30:00    2021-03-23 19:00:00
20        IG02          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed            31  8.576703e+08  2021-03-23 18:30:00  2021-03-23 19:00:00               31    2021-03-23 18:30:00    2021-03-23 19:00:00
21        IG02        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed            31  5.854114e+07  2021-03-23 18:30:00  2021-03-23 19:00:00               31    2021-03-23 18:30:00    2021-03-23 19:00:00
1130  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...            31  3.000000e+00  2021-03-23 18:30:00  2021-03-23 19:00:00               31    2021-03-23 18:30:00    2021-03-23 19:00:00
1123  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...            31  0.000000e+00  2021-03-23 18:30:00  2021-03-23 19:00:00               31    2021-03-23 18:30:00    2021-03-23 19:00:00
1120  Tomcat04        Tomcat-Sessions_7441--_SESSIONActiveCounter            31  0.000000e+00  2021-03-23 18:30:00  2021-03-23 19:00:00               31    2021-03-23 18:30:00    2021-03-23 19:00:00
1121  Tomcat04     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter            31  0.000000e+00  2021-03-23 18:30:00  2021-03-23 19:00:00               31    2021-03-23 18:30:00    2021-03-23 19:00:00
1122  Tomcat04     Tomcat-Sessions_7441--_SESSIONRejectedSessions            31  0.000000e+00  2021-03-23 18:30:00  2021-03-23 19:00:00               31    2021-03-23 18:30:00    2021-03-23 19:00:00
1125  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...            31  0.000000e+00  2021-03-23 18:30:00  2021-03-23 19:00:00               31    2021-03-23 18:30:00    2021-03-23 19:00:00
1124  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...            31  0.000000e+00  2021-03-23 18:30:00  2021-03-23 19:00:00               31    2021-03-23 18:30:00    2021-03-23 19:00:00
1128  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...            31  0.000000e+00  2021-03-23 18:30:00  2021-03-23 19:00:00               31    2021-03-23 18:30:00    2021-03-23 19:00:00, 19671)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# reuse df_metric and tz if available, otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Incident window (Asia/Shanghai) -> convert to UTC epoch seconds
start_local = pd.Timestamp('2021-03-23 18:30:00', tz=tz)
end_local   = pd.Timestamp('2021-03-23 19:00:00', tz=tz)
start_epoch = int(start_local.tz_convert('UTC').timestamp())
end_epoch   = int(end_local.tz_convert('UTC').timestamp())

# 1) Compute global_p95 and total_points per group using full series (rule 9)
global_agg = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        total_points=('value', 'count'),
        global_p95=('value', lambda s: float(s.quantile(0.95)))
    )
)

# 2) Filter incident-window rows
df_incident = df_metric[(df_metric['timestamp'] >= start_epoch) & (df_metric['timestamp'] <= end_epoch)].copy()

# 3) Incident points per group (only groups with at least one incident point)
incident_counts = (
    df_incident
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(incident_points=('value', 'count'))
)

# 4) Consider only groups that have at least one incident point by merging with global_agg
groups_in_window = pd.merge(global_agg, incident_counts, on=['cmdb_id','kpi_name'], how='inner')

# 5) Identify anomaly points within incident window: value > global_p95 (global threshold computed above)
# Merge global_p95 into incident rows
df_inc_with_p95 = pd.merge(
    df_incident,
    global_agg[['cmdb_id','kpi_name','global_p95']],
    on=['cmdb_id','kpi_name'],
    how='left'
)

# Select anomaly rows (value strictly greater than global_p95)
df_anomalies = df_inc_with_p95[df_inc_with_p95['value'] > df_inc_with_p95['global_p95']].copy()

# 6) Aggregate anomaly info per group
if not df_anomalies.empty:
    anomaly_agg = (
        df_anomalies
        .groupby(['cmdb_id','kpi_name'], as_index=False)
        .agg(
            anomaly_count=('value','count'),
            earliest_anomaly_timestamp=('timestamp','min'),
            max_anomaly_value=('value','max')
        )
    )
else:
    anomaly_agg = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp','max_anomaly_value'])

# 7) Merge groups_in_window with anomaly_agg to produce final per-group table (only groups that had incident points)
result = pd.merge(groups_in_window, anomaly_agg, on=['cmdb_id','kpi_name'], how='left')

# Fill missing anomaly_count with 0
result['anomaly_count'] = result['anomaly_count'].fillna(0).astype(int)
# earliest_anomaly_timestamp and max_anomaly_value remain NaN where anomaly_count==0

# 8) Compute severity_ratio = max_anomaly_value / global_p95 (only meaningful when anomaly_count>0)
result['severity_ratio'] = np.where(
    result['anomaly_count'] > 0,
    result['max_anomaly_value'] / result['global_p95'],
    np.nan
)

# 9) Convert earliest_anomaly_timestamp to Asia/Shanghai readable string (UTC+8) for display
def epoch_to_local_str(v):
    if pd.isna(v):
        return None
    return pd.to_datetime(int(v), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S')

result['earliest_anomaly_timestamp'] = result['earliest_anomaly_timestamp'].apply(lambda x: epoch_to_local_str(x) if pd.notna(x) else None)

# 10) Keep requested columns and filter to groups with anomaly_count>0, sort by severity_ratio desc, limit top 20
selected_cols = [
    'cmdb_id','kpi_name','total_points','global_p95',
    'incident_points','anomaly_count','earliest_anomaly_timestamp','max_anomaly_value','severity_ratio'
]
anomalies_table = result[result['anomaly_count'] > 0].sort_values(by='severity_ratio', ascending=False)[selected_cols].head(20)

# 11) Overall number of groups that have anomaly_count>0
anomalous_groups_count = int(result[result['anomaly_count'] > 0].shape[0])

# Display compact outputs: anomalies_table (top 20) and anomalous_groups_count
anomalies_table, anomalous_groups_count

```
Out[3]:
```
```python
# Build a concise plain-English summary using the previously computed results:
# anomalies_table (top groups sorted by severity_ratio) and anomalous_groups_count are reused from session.

try:
    anomalies_table
    anomalous_groups_count
except NameError:
    raise RuntimeError("Required variables not found in the session.")

# Prepare example lines from the top 5 groups
examples = []
for r in anomalies_table.head(5).itertuples(index=False):
    # r fields: cmdb_id, kpi_name, total_points, global_p95, incident_points,
    # anomaly_count, earliest_anomaly_timestamp, max_anomaly_value, severity_ratio
    sr = r.severity_ratio
    sr_str = "inf" if (pd.isna(sr) and r.anomaly_count>0) or (pd.notna(sr) and (str(sr) == 'inf' or sr == float('inf'))) else f"{float(sr):.3g}"
    examples.append(
        f"- {r.cmdb_id} | {r.kpi_name} | anomaly_count={int(r.anomaly_count)} | "
        f"earliest={r.earliest_anomaly_timestamp} | max_value={float(r.max_anomaly_value):.6g} | severity_ratio={sr_str}"
    )

summary_text = (
    "Summary of detected anomalies in metric data (incident window: 2021-03-23 18:30–19:00 UTC+8):\n\n"
    f"1) Total number of metric groups with anomaly_count>0: {anomalous_groups_count}\n\n"
    "2) Notes on methodology:\n"
    "- global_p95 is the 95th percentile computed over each group's full available series.\n"
    "- Anomaly points are incident-window points with value > global_p95.\n"
    "- severity_ratio = max_anomaly_value / global_p95 (undefined/infinite when global_p95 == 0).\n\n"
    "3) Top examples (by severity_ratio):\n"
    + "\n".join(examples) + "\n\n"
    "Observations:\n"
    "- Several top groups show global_p95 == 0, producing infinite severity_ratio (e.g., TCP CLOSE-WAIT, TCP FIN-WAIT, MySQL row-lock-waits with a single incident spike).\n"
    "- High finite severity ratios include elevated CPU WIO on Tomcat hosts (severity_ratio ~4–8) and several disk I/O metrics with ratios ~3–4.\n\n"
    "The returned table contains the requested aggregated columns for the top groups (cmdb_id, kpi_name, total_points, global_p95, "
    "incident_points, anomaly_count, earliest_anomaly_timestamp, max_anomaly_value, severity_ratio)."
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name  total_points  global_p95  incident_points  anomaly_count earliest_anomaly_timestamp  max_anomaly_value  severity_ratio
223       MG02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT            30    0.000000               30              1        2021-03-23 18:35:00           1.000000             inf
119       MG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT            29    0.000000               29              1        2021-03-23 18:40:00           1.000000             inf
455    Mysql02     Mysql-MySQL_3306_Innodb row lock current waits            29    0.000000               29              1        2021-03-23 18:52:00           1.000000             inf
1022  Tomcat04                             OSLinux-CPU_CPU_CPUWio            30    0.622215               30              2        2021-03-23 18:30:00           4.865876        7.820249
780   Tomcat02                             OSLinux-CPU_CPU_CPUWio            30    0.561455               30              2        2021-03-23 18:36:00           3.564892        6.349376
901   Tomcat03                             OSLinux-CPU_CPU_CPUWio            30    0.684744               30              2        2021-03-23 18:33:00           3.286326        4.799352
662   Tomcat01                             OSLinux-CPU_CPU_CPUWio            30    0.723021               30              2        2021-03-23 18:45:00           3.196841        4.421504
715   Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead            16  311.566667               16              1        2021-03-23 19:00:00        1246.266667        4.000000
703   Tomcat01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps            16    3.750000               16              1        2021-03-23 18:50:00          15.000000        4.000000
714   Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps            16    0.758333               16              1        2021-03-23 19:00:00           3.033333        4.000000
706   Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead            16   97.700000               16              1        2021-03-23 19:00:00         372.800000        3.815763
705   Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps            16    2.979167               16              1        2021-03-23 19:00:00          11.266667        3.781818
48        MG01                    OSLinux-CPU_CPU-2_SingleCpuidle            15    0.219342               15              1        2021-03-23 18:55:00           0.731140        3.333333
570    Redis02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...            15    5.280000               15              1        2021-03-23 18:56:00          17.600000        3.333333
830   Tomcat02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps            15    0.600000               15              1        2021-03-23 18:50:00           2.000000        3.333333
1017  Tomcat04                    OSLinux-CPU_CPU-3_SingleCpuidle            15   29.826994               15              1        2021-03-23 18:42:00          98.919548        3.316444
944   Tomcat03    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps            15    1.873343               15              1        2021-03-23 18:30:00           5.816700        3.104984
1065  Tomcat04    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps            15    1.951690               15              1        2021-03-23 18:30:00           6.000000        3.074259
55        MG01                             OSLinux-CPU_CPU_CPUWio            30    1.030986               30              2        2021-03-23 18:35:00           2.856905        2.771040
945   Tomcat03    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead            15   20.400000               15              1        2021-03-23 18:30:00          54.000000        2.647059, 463)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse previously loaded data if available, otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure global_p95 and incident-window anomaly rows are available; recompute if needed
try:
    global_agg
except NameError:
    global_agg = (
        df_metric
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)
        .agg(
            total_points=('value', 'count'),
            global_p95=('value', lambda s: float(s.quantile(0.95)))
        )
    )

# Incident window (Asia/Shanghai) -> UTC epoch seconds
start_local = pd.Timestamp('2021-03-23 18:30:00', tz=tz)
end_local   = pd.Timestamp('2021-03-23 19:00:00', tz=tz)
start_epoch = int(start_local.tz_convert('UTC').timestamp())
end_epoch   = int(end_local.tz_convert('UTC').timestamp())

# Incident-window rows
df_incident = df_metric[(df_metric['timestamp'] >= start_epoch) & (df_metric['timestamp'] <= end_epoch)].copy()

# Merge global_p95 into incident rows
df_inc_with_p95 = pd.merge(
    df_incident,
    global_agg[['cmdb_id','kpi_name','global_p95','total_points']],
    on=['cmdb_id','kpi_name'],
    how='left'
)

# Anomaly rows (value > global_p95)
df_anomalies = df_inc_with_p95[df_inc_with_p95['value'] > df_inc_with_p95['global_p95']].copy()

# If no anomalies, return empty structures
if df_anomalies.empty:
    final_table = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','total_points','global_p95','incident_points',
        'number_of_faults','longest_fault_length_minutes',
        'first_fault_start_timestamp','first_fault_end_timestamp',
        'max_value_within_first_fault','severity_ratio_first_fault'
    ])
    unique_cmdb_ids_top20 = set()
else:
    # For incident_points per group (needed in output), compute counts
    incident_counts = (
        df_incident
        .groupby(['cmdb_id','kpi_name'], as_index=False)
        .agg(incident_points=('value','count'))
    )

    # We'll analyze fault segments per group from df_anomalies
    # A fault segment is consecutive anomaly timestamps with diff == 60 seconds
    segments_list = []
    grouped = df_anomalies.sort_values(['cmdb_id','kpi_name','timestamp']).groupby(['cmdb_id','kpi_name'])
    for (cmdb, kpi), g in grouped:
        g = g[['timestamp','value']].reset_index(drop=True)
        # compute breaks where diff != 60
        diffs = g['timestamp'].diff().fillna(60)  # first row treated as new segment
        new_seg = (diffs != 60).cumsum()  # segment id
        g['seg_id'] = new_seg
        segs = g.groupby('seg_id', as_index=False).agg(
            seg_start_ts=('timestamp','min'),
            seg_end_ts=('timestamp','max'),
            seg_len_points=('timestamp','count'),
            seg_max_value=('value','max')
        ).sort_values('seg_start_ts')
        number_of_faults = int(segs.shape[0])
        longest_fault_length_minutes = int(segs['seg_len_points'].max())
        # first fault is the earliest by seg_start_ts
        first = segs.iloc[0]
        first_start = int(first['seg_start_ts'])
        first_end = int(first['seg_end_ts'])
        max_value_first = float(first['seg_max_value'])
        segments_list.append({
            'cmdb_id': cmdb,
            'kpi_name': kpi,
            'number_of_faults': number_of_faults,
            'longest_fault_length_minutes': longest_fault_length_minutes,
            'first_fault_start_timestamp': first_start,
            'first_fault_end_timestamp': first_end,
            'max_value_within_first_fault': max_value_first
        })

    segments_df = pd.DataFrame(segments_list)

    # Merge with global stats and incident counts
    merged = segments_df.merge(global_agg, on=['cmdb_id','kpi_name'], how='left')
    merged = merged.merge(incident_counts, on=['cmdb_id','kpi_name'], how='left')

    # Compute severity_ratio_first_fault
    # If global_p95 == 0 and max_value_within_first_fault > 0 -> inf
    merged['severity_ratio_first_fault'] = merged.apply(
        lambda r: (float('inf') if (r['global_p95'] == 0 and r['max_value_within_first_fault'] > 0)
                   else (r['max_value_within_first_fault'] / r['global_p95']) ),
        axis=1
    )

    # Convert first fault timestamps to Asia/Shanghai strings
    def epoch_to_local_str(v):
        if pd.isna(v):
            return None
        return pd.to_datetime(int(v), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S')

    merged['first_fault_start_timestamp'] = merged['first_fault_start_timestamp'].apply(epoch_to_local_str)
    merged['first_fault_end_timestamp'] = merged['first_fault_end_timestamp'].apply(epoch_to_local_str)

    # Select and order columns
    final_table = merged[[
        'cmdb_id','kpi_name','total_points','global_p95','incident_points',
        'number_of_faults','longest_fault_length_minutes',
        'first_fault_start_timestamp','first_fault_end_timestamp',
        'max_value_within_first_fault','severity_ratio_first_fault'
    ]].sort_values(by='severity_ratio_first_fault', ascending=False).head(20)

    unique_cmdb_ids_top20 = set(final_table['cmdb_id'].unique())

# Output compact results: final_table (top 20) and unique_cmdb_ids_top20
final_table, unique_cmdb_ids_top20

```
Out[4]:
```
Summary of fault-segment detection (top 20 groups by severity_ratio_first_fault):

Highlights:
- Three groups have global_p95 == 0, producing infinite severity_ratio for the first fault:
  1) MG01 — OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT
     - total_points: 29, incident_points: 29, number_of_faults: 1
     - first fault: 2021-03-23 18:40:00 to 18:40:00, max_value_in_first_fault: 1.0, severity_ratio_first_fault: inf
  2) Mysql02 — Mysql-MySQL_3306_Innodb row lock current waits
     - total_points: 29, incident_points: 29, number_of_faults: 1
     - first fault: 2021-03-23 18:52:00 to 18:52:00, max_value: 1.0, severity_ratio_first_fault: inf
  3) MG02 — OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT
     - total_points: 30, incident_points: 30, number_of_faults: 1
     - first fault: 2021-03-23 18:35:00 to 18:35:00, max_value: 1.0, severity_ratio_first_fault: inf

- High finite severity examples (first faults):
  - Tomcat01 — sdb_DSKRead: total_points 16, global_p95 311.566667, incident_points 16, first fault at 2021-03-23 19:00:00, max_value_in_first_fault 1246.266667, severity_ratio_first_fault = 4.0
  - Tomcat01 — sdb_DSKRTps: total_points 16, global_p95 0.758333, incident_points 16, first fault 19:00:00, max_value 3.033333, severity_ratio_first_fault = 4.0
  - Tomcat01 — sda_DSKBps: total_points 16, global_p95 3.75, incident_points 16, first fault 18:50:00, max_value 15.0, severity_ratio_first_fault = 4.0
  - Several Tomcat and Tomcat-cluster disk/CPU metrics show severity ratios in the ~2.3–4.0 range.

General observations from the top-20:
- Most fault segments are short single-minute spikes: number_of_faults = 1 and longest_fault_length_minutes = 1 for the majority of top entries.
- Incident coverage for these top groups is substantial (many have incident_points equal to their per-group incident-window counts like 15–30).
- The highest-severity items are either (a) metrics with global_p95 == 0 where any positive spike yields infinite ratio, or (b) disk/CPU metrics showing multi-fold increases over their group-level 95th percentile.

Unique cmdb_id values present in the top-20 table:
{'MG01', 'Tomcat01', 'Tomcat04', 'Tomcat02', 'MG02', 'Tomcat03', 'Redis02', 'Mysql02'}

The returned table contains the requested aggregated columns for each top group: cmdb_id, kpi_name, total_points, global_p95, incident_points, number_of_faults, longest_fault_length_minutes, first_fault_start_timestamp, first_fault_end_timestamp, max_value_within_first_fault, severity_ratio_first_fault.

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                                           kpi_name  total_points  global_p95  incident_points  number_of_faults  longest_fault_length_minutes first_fault_start_timestamp first_fault_end_timestamp  max_value_within_first_fault  severity_ratio_first_fault
59       MG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT            29    0.000000               29                 1                             1         2021-03-23 18:40:00       2021-03-23 18:40:00                      1.000000                         inf
201   Mysql02     Mysql-MySQL_3306_Innodb row lock current waits            29    0.000000               29                 1                             1         2021-03-23 18:52:00       2021-03-23 18:52:00                      1.000000                         inf
105      MG02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT            30    0.000000               30                 1                             1         2021-03-23 18:35:00       2021-03-23 18:35:00                      1.000000                         inf
296  Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps            16    0.758333               16                 1                             1         2021-03-23 19:00:00       2021-03-23 19:00:00                      3.033333                    4.000000
297  Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead            16  311.566667               16                 1                             1         2021-03-23 19:00:00       2021-03-23 19:00:00                   1246.266667                    4.000000
288  Tomcat01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps            16    3.750000               16                 1                             1         2021-03-23 18:50:00       2021-03-23 18:50:00                     15.000000                    4.000000
291  Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead            16   97.700000               16                 1                             1         2021-03-23 19:00:00       2021-03-23 19:00:00                    372.800000                    3.815763
290  Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps            16    2.979167               16                 1                             1         2021-03-23 19:00:00       2021-03-23 19:00:00                     11.266667                    3.781818
237   Redis02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...            15    5.280000               15                 1                             1         2021-03-23 18:56:00       2021-03-23 18:56:00                     17.600000                    3.333333
27       MG01                    OSLinux-CPU_CPU-2_SingleCpuidle            15    0.219342               15                 1                             1         2021-03-23 18:55:00       2021-03-23 18:55:00                      0.731140                    3.333333
340  Tomcat02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps            15    0.600000               15                 1                             1         2021-03-23 18:50:00       2021-03-23 18:50:00                      2.000000                    3.333333
413  Tomcat04                    OSLinux-CPU_CPU-3_SingleCpuidle            15   29.826994               15                 1                             1         2021-03-23 18:42:00       2021-03-23 18:42:00                     98.919548                    3.316444
380  Tomcat03    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps            15    1.873343               15                 1                             1         2021-03-23 18:30:00       2021-03-23 18:30:00                      5.816700                    3.104984
426  Tomcat04    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps            15    1.951690               15                 1                             1         2021-03-23 18:30:00       2021-03-23 18:30:00                      6.000000                    3.074259
381  Tomcat03    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead            15   20.400000               15                 1                             1         2021-03-23 18:30:00       2021-03-23 18:30:00                     54.000000                    2.647059
427  Tomcat04    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead            15   19.479990               15                 1                             1         2021-03-23 18:30:00       2021-03-23 18:30:00                     50.933300                    2.614647
425  Tomcat04     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps            15    2.500000               15                 1                             1         2021-03-23 18:42:00       2021-03-23 18:42:00                      6.000000                    2.400000
378  Tomcat03     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps            15    2.200000               15                 1                             1         2021-03-23 18:46:00       2021-03-23 18:46:00                      5.000000                    2.272727
333  Tomcat02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps            15    2.200000               15                 1                             1         2021-03-23 18:46:00       2021-03-23 18:46:00                      5.000000                    2.272727
379  Tomcat03  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...            15    0.630000               15                 1                             1         2021-03-23 18:30:00       2021-03-23 18:30:00                      1.400000                    2.222222, {'MG01', 'Tomcat01', 'Tomcat04', 'Tomcat02', 'MG02', 'Tomcat03', 'Redis02', 'Mysql02'})```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# reuse variables if present
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# 1) Global aggregation by (cmdb_id, trace_name) using full series (global_p95 must be computed on full series)
global_trace_agg = (
    df_trace
    .groupby(['cmdb_id','trace_name'], as_index=False)
    .agg(
        total_points=('value','count'),
        global_p95=('value', lambda s: float(s.quantile(0.95)))
    )
)

# 2) Incident window (Asia/Shanghai) -> convert to UTC epoch seconds
start_local = pd.Timestamp('2021-03-23 18:30:00', tz=tz)
end_local   = pd.Timestamp('2021-03-23 19:00:00', tz=tz)
start_epoch = int(start_local.tz_convert('UTC').timestamp())
end_epoch   = int(end_local.tz_convert('UTC').timestamp())

# 3) Filter trace rows to incident window
df_trace_incident = df_trace[(df_trace['timestamp'] >= start_epoch) & (df_trace['timestamp'] <= end_epoch)].copy()

# 4) Incident points per group
incident_counts = (
    df_trace_incident
    .groupby(['cmdb_id','trace_name'], as_index=False)
    .agg(incident_points=('value','count'))
)

# 5) Merge global_p95 into incident rows and identify anomaly rows (value > global_p95)
df_inc_with_p95 = pd.merge(
    df_trace_incident,
    global_trace_agg[['cmdb_id','trace_name','global_p95','total_points']],
    on=['cmdb_id','trace_name'],
    how='left'
)

df_trace_anomalies = df_inc_with_p95[df_inc_with_p95['value'] > df_inc_with_p95['global_p95']].copy()

# 6) Aggregate anomaly info per group
if not df_trace_anomalies.empty:
    anomaly_agg = (
        df_trace_anomalies
        .groupby(['cmdb_id','trace_name'], as_index=False)
        .agg(
            anomaly_count=('value','count'),
            earliest_anomaly_timestamp=('timestamp','min'),
            max_anomaly_value=('value','max')
        )
    )
else:
    anomaly_agg = pd.DataFrame(columns=['cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp','max_anomaly_value'])

# 7) Merge aggregates: keep only groups with anomaly_count > 0 by inner joining anomaly_agg with global and incident counts
merged = pd.merge(anomaly_agg, global_trace_agg, on=['cmdb_id','trace_name'], how='left')
merged = pd.merge(merged, incident_counts, on=['cmdb_id','trace_name'], how='left')

# 8) Compute severity_ratio (handle global_p95 == 0)
def compute_severity(row):
    gp = row['global_p95']
    mv = row['max_anomaly_value']
    if pd.isna(gp):
        return np.nan
    if gp == 0:
        return float('inf') if mv > 0 else np.nan
    return mv / gp

merged['severity_ratio'] = merged.apply(compute_severity, axis=1)

# 9) Convert earliest_anomaly_timestamp to Asia/Shanghai readable strings
def epoch_to_local_str(v):
    if pd.isna(v):
        return None
    return pd.to_datetime(int(v), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S')

merged['earliest_anomaly_timestamp'] = merged['earliest_anomaly_timestamp'].apply(lambda x: epoch_to_local_str(x) if pd.notna(x) else None)

# 10) Prepare final compact table sorted by earliest_anomaly_timestamp ascending, limit top 20
selected_cols = [
    'cmdb_id','trace_name','total_points','global_p95',
    'incident_points','anomaly_count','earliest_anomaly_timestamp',
    'max_anomaly_value','severity_ratio'
]
# ensure we only include rows with anomaly_count > 0
merged['anomaly_count'] = merged['anomaly_count'].fillna(0).astype(int)
trace_anomalies_table = merged[merged['anomaly_count'] > 0].sort_values(by='earliest_anomaly_timestamp', ascending=True)[selected_cols].head(20)

# 11) Overall count of trace groups with anomaly_count>0 and set of unique cmdb_id in returned table
overall_anomalous_groups_count = int(merged[merged['anomaly_count'] > 0].shape[0])
unique_cmdb_ids_in_top20 = set(trace_anomalies_table['cmdb_id'].unique())

# Display compact outputs
trace_anomalies_table, overall_anomalous_groups_count, unique_cmdb_ids_in_top20

```
Out[5]:
```
summary = (
    "Trace anomaly summary (incident window 2021-03-23 18:30–19:00 UTC+8):\n\n"
    "- Total trace groups with anomaly_count>0: 228\n"
    "- Unique cmdb_id values in the top-20 table: {'MG01', 'dockerB2', 'dockerB1', 'IG02', 'Tomcat03', 'Tomcat02', 'dockerA2', 'MG02'}\n\n"
    "- Earliest anomalies: the first anomalies appear at 2021-03-23 18:33:00 (UTC+8) for dockerA2 <-> MG02 traces, then a cluster at 18:34:00 for many groups.\n\n"
    "- Severity and magnitude (top-20):\n"
    "  * Severity ratios in the top-20 are modest, roughly from ~1.025 to ~1.162 (i.e., 2.5%–16% above each group's global 95th percentile).\n"
    "  * Typical pattern: incident_points = 30–31, anomaly_count = 2 (two incident-window spikes), max_anomaly_value a bit above global_p95.\n"
    "  * Examples from the top-20:\n"
    "    - dockerA2 trace.to_MG02.row_count: total_points=31, global_p95=59.0, incident_points=31, anomaly_count=2, earliest=2021-03-23 18:33:00, max=62.0, severity=~1.051\n"
    "    - dockerB1 trace.to_MG01.row_count: total_points=31, global_p95=217.5, incident_points=31, anomaly_count=2, earliest=2021-03-23 18:34:00, max=252.0, severity=~1.159\n"
    "    - Tomcat02 trace.from_IG02.row_count: total_points=31, global_p95=145.5, incident_points=31, anomaly_count=2, earliest=2021-03-23 18:34:00, max=150.0, severity=~1.031\n\n"
    "- Interpretation: many trace metrics across multiple services show small but consistent spikes starting ~18:33–18:34. "
    "The deviations are generally modest (near 1.03–1.16× the group-level p95), suggesting short transient load/traffic increases rather than very large outliers. "
    "Investigation should focus on the listed components (MG01/MG02, dockerA2/dockerB1/dockerB2, IG02, Tomcat02/03) and the trace paths between them around 18:33–18:34 UTC+8.\n"
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                     trace_name  total_points  global_p95  incident_points  anomaly_count earliest_anomaly_timestamp  max_anomaly_value  severity_ratio
197  dockerA2        trace.to_MG02.row_count            31        59.0               31              2        2021-03-23 18:33:00               62.0        1.050847
86       MG02  trace.from_dockerA2.row_count            31        59.0               31              2        2021-03-23 18:33:00               62.0        1.050847
209  dockerB1        trace.to_MG01.row_count            31       217.5               31              2        2021-03-23 18:34:00              252.0        1.158621
50       MG01  trace.from_dockerB1.row_count            31       217.5               31              2        2021-03-23 18:34:00              252.0        1.158621
128  Tomcat02      trace.from_IG02.row_count            31       145.5               31              2        2021-03-23 18:34:00              150.0        1.030928
56       MG01           trace.self.row_count            31       844.0               31              2        2021-03-23 18:34:00              885.0        1.048578
206  dockerB1           trace.self.row_count            31      3536.5               31              2        2021-03-23 18:34:00             4040.0        1.142372
203  dockerB1      trace.from_MG02.row_count            31       213.5               31              2        2021-03-23 18:34:00              248.0        1.161593
200  dockerB1      trace.from_MG01.row_count            31       205.5               31              2        2021-03-23 18:34:00              221.0        1.075426
26       IG02    trace.to_Tomcat03.row_count            31       137.5               31              2        2021-03-23 18:34:00              141.0        1.025455
65       MG01    trace.to_dockerB1.row_count            31       205.5               31              2        2021-03-23 18:34:00              221.0        1.075426
23       IG02    trace.to_Tomcat02.row_count            31       145.5               31              2        2021-03-23 18:34:00              150.0        1.030928
68       MG01    trace.to_dockerB2.row_count            31        72.0               31              2        2021-03-23 18:34:00               77.0        1.069444
215  dockerB2      trace.from_MG01.row_count            31        72.0               31              2        2021-03-23 18:34:00               77.0        1.069444
17       IG02           trace.self.row_count            31      1077.0               31              2        2021-03-23 18:34:00             1126.0        1.045497
104      MG02    trace.to_dockerB1.row_count            31       213.5               31              2        2021-03-23 18:34:00              248.0        1.161593
221  dockerB2           trace.self.row_count            31      1139.5               31              2        2021-03-23 18:34:00             1229.0        1.078543
107      MG02    trace.to_dockerB2.row_count            30        76.7               30              2        2021-03-23 18:34:00               84.0        1.095176
218  dockerB2      trace.from_MG02.row_count            30        76.7               30              2        2021-03-23 18:34:00               84.0        1.095176
143  Tomcat03      trace.from_IG02.row_count            31       137.5               31              2        2021-03-23 18:34:00              141.0        1.025455, 228, {'MG01', 'dockerB2', 'dockerB1', 'IG02', 'Tomcat03', 'Tomcat02', 'dockerA2', 'MG02'})```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

# reuse existing variables if available, otherwise load
try:
    df_log
except NameError:
    df_log = pd.read_csv('log.csv')
try:
    df_error
except NameError:
    df_error = pd.read_csv('error_logs.csv')

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Incident window (Asia/Shanghai) -> UTC epoch seconds
start_local = pd.Timestamp('2021-03-23 18:30:00', tz=tz)
end_local   = pd.Timestamp('2021-03-23 19:00:00', tz=tz)
start_epoch = int(start_local.tz_convert('UTC').timestamp())
end_epoch   = int(end_local.tz_convert('UTC').timestamp())

# ---- Part 1: log.csv analysis ----
# 1) Global aggregation by (cmdb_id, log_name) using full series
log_global = (
    df_log
    .groupby(['cmdb_id','log_name'], as_index=False)
    .agg(
        total_points=('value','count'),
        global_p95=('value', lambda s: float(s.quantile(0.95)))
    )
)

# 2) Filter to incident window
df_log_incident = df_log[(df_log['timestamp'] >= start_epoch) & (df_log['timestamp'] <= end_epoch)].copy()

# 3) incident_points per group
log_inc_counts = (
    df_log_incident
    .groupby(['cmdb_id','log_name'], as_index=False)
    .agg(incident_points=('value','count'))
)

# 4) Merge global thresholds into incident rows and detect anomalies (value > global_p95)
df_log_inc_with_p95 = pd.merge(
    df_log_incident,
    log_global[['cmdb_id','log_name','global_p95','total_points']],
    on=['cmdb_id','log_name'],
    how='left'
)

df_log_anomalies = df_log_inc_with_p95[df_log_inc_with_p95['value'] > df_log_inc_with_p95['global_p95']].copy()

# 5) Aggregate anomaly info per group
if not df_log_anomalies.empty:
    log_anom_agg = (
        df_log_anomalies
        .groupby(['cmdb_id','log_name'], as_index=False)
        .agg(
            anomaly_count=('value','count'),
            earliest_anomaly_timestamp=('timestamp','min'),
            max_anomaly_value=('value','max')
        )
    )
else:
    log_anom_agg = pd.DataFrame(columns=['cmdb_id','log_name','anomaly_count','earliest_anomaly_timestamp','max_anomaly_value'])

# 6) Merge aggregates: keep only groups with anomaly_count>0
log_merged = pd.merge(log_anom_agg, log_global, on=['cmdb_id','log_name'], how='left')
log_merged = pd.merge(log_merged, log_inc_counts, on=['cmdb_id','log_name'], how='left')

# 7) Compute severity_ratio (handle global_p95 == 0 -> inf when max_anomaly_value > 0)
def compute_severity_log(row):
    gp = row['global_p95']
    mv = row['max_anomaly_value']
    if pd.isna(gp):
        return np.nan
    if gp == 0:
        return float('inf') if mv > 0 else np.nan
    return mv / gp

if not log_merged.empty:
    log_merged['severity_ratio'] = log_merged.apply(compute_severity_log, axis=1)
    # Convert earliest_anomaly_timestamp to Asia/Shanghai strings
    def epoch_to_local_str(v):
        if pd.isna(v):
            return None
        return pd.to_datetime(int(v), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S')
    log_merged['earliest_anomaly_timestamp'] = log_merged['earliest_anomaly_timestamp'].apply(lambda x: epoch_to_local_str(x) if pd.notna(x) else None)
else:
    log_merged['severity_ratio'] = pd.Series(dtype=float)

# 8) Final compact table: only groups with anomaly_count>0, sorted by earliest_anomaly_timestamp asc, top 20
log_merged['anomaly_count'] = log_merged['anomaly_count'].fillna(0).astype(int)
log_result_all = log_merged[log_merged['anomaly_count'] > 0].copy()
log_result_table = log_result_all.sort_values(by='earliest_anomaly_timestamp', ascending=True)[[
    'cmdb_id','log_name','total_points','global_p95','incident_points',
    'anomaly_count','earliest_anomaly_timestamp','max_anomaly_value','severity_ratio'
]].head(20)

log_groups_with_anomaly_count = int(log_result_all.shape[0])
unique_log_cmdb_ids_top20 = set(log_result_table['cmdb_id'].unique())

# ---- Part 2: error_logs.csv ----
# Check if file contains any rows
error_rows = int(df_error.shape[0])
if error_rows == 0:
    error_summary_text = "error_logs.csv contains zero rows."
    error_top10 = pd.DataFrame()  # empty placeholder
else:
    # compute top 10 distinct messages with counts and earliest timestamp per message
    err_agg = (
        df_error
        .groupby('message', as_index=False)
        .agg(count=('message','count'), earliest_timestamp=('timestamp','min'))
        .sort_values(by='count', ascending=False)
        .head(10)
    )
    # convert earliest_timestamp to Asia/Shanghai string
    def epoch_to_local_str_err(v):
        if pd.isna(v):
            return None
        return pd.to_datetime(int(v), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S')
    err_agg['earliest_timestamp'] = err_agg['earliest_timestamp'].apply(epoch_to_local_str_err)
    error_top10 = err_agg[['message','count','earliest_timestamp']]

    error_summary_text = f"error_logs.csv contains {error_rows} rows; top 10 messages returned."

# Final compact outputs
log_result_table, log_groups_with_anomaly_count, unique_log_cmdb_ids_top20, error_summary_text, error_top10

```
Out[6]:
```
Log analysis (incident window 2021-03-23 18:30–19:00 UTC+8)

- Total number of log groups with anomaly_count > 0: 10
- Unique cmdb_id values in the returned top-20 table: {'apache02', 'Tomcat01', 'Tomcat04', 'apache01', 'Tomcat03', 'Tomcat02', 'Mysql01', 'Mysql02'}

Top anomalous log groups (sorted by earliest anomaly time; shown fields: cmdb_id | log_name | total_points | global_p95 | incident_points | anomaly_count | earliest_anomaly_timestamp | max_anomaly_value | severity_ratio)
- Tomcat02 | log.row_count | total_points=31 | global_p95=282.50 | incident_points=31 | anomaly_count=2 | earliest=2021-03-23 18:34:00 | max=296.00 | severity≈1.048
- Tomcat04 | log.row_count | total_points=31 | global_p95=282.00 | incident_points=31 | anomaly_count=2 | earliest=2021-03-23 18:35:00 | max=283.00 | severity≈1.004
- Mysql01  | log.row_count | total_points=30 | global_p95=37.55  | incident_points=30 | anomaly_count=2 | earliest=2021-03-23 18:37:00 | max=39.00  | severity≈1.039
- Tomcat03 | log.row_count | total_points=31 | global_p95=272.50 | incident_points=31 | anomaly_count=2 | earliest=2021-03-23 18:39:00 | max=278.00 | severity≈1.020
- Tomcat04 | log.gc_pause_max| total_points=14| global_p95=0.0935 | incident_points=14 | anomaly_count=1 | earliest=2021-03-23 18:43:00 | max=0.10  | severity≈1.070
- apache02  | log.row_count | total_points=31 | global_p95=549.50 | incident_points=31 | anomaly_count=2 | earliest=2021-03-23 18:44:00 | max=560.00 | severity≈1.019
- Tomcat03 | log.gc_pause_max| total_points=12| global_p95=0.1035 | incident_points=12 | anomaly_count=1 | earliest=2021-03-23 18:45:00 | max=0.12  | severity≈1.159
- apache01  | log.row_count | total_points=31 | global_p95=543.00 | incident_points=31 | anomaly_count=2 | earliest=2021-03-23 18:49:00 | max=568.00 | severity≈1.046
- Tomcat01 | log.row_count | total_points=31 | global_p95=273.50 | incident_points=31 | anomaly_count=2 | earliest=2021-03-23 18:50:00 | max=315.00 | severity≈1.152
- Mysql02  | log.row_count | total_points=2  | global_p95=1213.60| incident_points=2  | anomaly_count=1 | earliest=2021-03-23 18:59:00 | max=1225.00| severity≈1.009

Notes:
- All severity ratios are modest (≈1.00–1.16), indicating small to moderate spikes above each group's global 95th percentile.
- Many affected groups are log.row_count metrics across Tomcat/apache/mysql hosts; a couple of gc_pause_max and a Mysql02 short series also show anomalies.

Error logs
- error_logs.csv contains zero rows (no error log messages present).

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id          log_name  total_points  global_p95  incident_points  anomaly_count earliest_anomaly_timestamp  max_anomaly_value  severity_ratio
3  Tomcat02     log.row_count            31    282.5000               31              2        2021-03-23 18:34:00             296.00        1.047788
7  Tomcat04     log.row_count            31    282.0000               31              2        2021-03-23 18:35:00             283.00        1.003546
0   Mysql01     log.row_count            30     37.5500               30              2        2021-03-23 18:37:00              39.00        1.038615
5  Tomcat03     log.row_count            31    272.5000               31              2        2021-03-23 18:39:00             278.00        1.020183
6  Tomcat04  log.gc_pause_max            14      0.0935               14              1        2021-03-23 18:43:00               0.10        1.069519
9  apache02     log.row_count            31    549.5000               31              2        2021-03-23 18:44:00             560.00        1.019108
4  Tomcat03  log.gc_pause_max            12      0.1035               12              1        2021-03-23 18:45:00               0.12        1.159420
8  apache01     log.row_count            31    543.0000               31              2        2021-03-23 18:49:00             568.00        1.046041
2  Tomcat01     log.row_count            31    273.5000               31              2        2021-03-23 18:50:00             315.00        1.151737
1   Mysql02     log.row_count             2   1213.6000                2              1        2021-03-23 18:59:00            1225.00        1.009394, 10, {'apache02', 'Tomcat01', 'Tomcat04', 'apache01', 'Tomcat03', 'Tomcat02', 'Mysql01', 'Mysql02'}, 'error_logs.csv contains zero rows.', Empty DataFrame
Columns: []
Index: [])```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

# Reuse loaded DataFrames if present, otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')
try:
    df_log
except NameError:
    df_log = pd.read_csv('log.csv')

tz = pytz.timezone('Asia/Shanghai')

# Candidate components
candidates = ['MG01','MG02','Tomcat01','Tomcat02','Tomcat03','Tomcat04','Mysql02','Redis02']

# Incident window (Asia/Shanghai) -> UTC epoch seconds
start_local = pd.Timestamp('2021-03-23 18:30:00', tz=tz)
end_local   = pd.Timestamp('2021-03-23 19:00:00', tz=tz)
start_epoch = int(start_local.tz_convert('UTC').timestamp())
end_epoch   = int(end_local.tz_convert('UTC').timestamp())

# -------------------------
# Metric: compute per-group global_p95 (full series) and detect fault segments in incident window
# -------------------------
metric_global = (
    df_metric
    .groupby(['cmdb_id','kpi_name'], as_index=False)
    .agg(total_points=('value','count'),
         global_p95=('value', lambda s: float(s.quantile(0.95))))
)

df_metric_inc = df_metric[(df_metric['timestamp'] >= start_epoch) & (df_metric['timestamp'] <= end_epoch)].copy()
df_metric_inc = pd.merge(df_metric_inc, metric_global[['cmdb_id','kpi_name','global_p95','total_points']], on=['cmdb_id','kpi_name'], how='left')

# anomaly points (value > global_p95)
df_metric_anom = df_metric_inc[df_metric_inc['value'] > df_metric_inc['global_p95']].copy()

metric_faults_list = []
if not df_metric_anom.empty:
    g = df_metric_anom.sort_values(['cmdb_id','kpi_name','timestamp']).groupby(['cmdb_id','kpi_name'])
    for (cmdb,kpi), grp in g:
        grp = grp[['timestamp','value']].reset_index(drop=True)
        diffs = grp['timestamp'].diff().fillna(60)
        seg_id = (diffs != 60).cumsum()
        grp['seg_id'] = seg_id
        segs = grp.groupby('seg_id', as_index=False).agg(
            seg_start_ts=('timestamp','min'),
            seg_end_ts=('timestamp','max'),
            seg_len_points=('timestamp','count'),
            seg_max_value=('value','max')
        ).sort_values('seg_start_ts')
        number_of_faults = int(segs.shape[0])
        longest_fault_length = int(segs['seg_len_points'].max())
        first = segs.iloc[0]
        first_start = int(first['seg_start_ts'])
        first_end = int(first['seg_end_ts'])
        max_first = float(first['seg_max_value'])
        # get global_p95 and total_points
        gp_row = metric_global[(metric_global['cmdb_id']==cmdb) & (metric_global['kpi_name']==kpi)]
        gp = float(gp_row['global_p95'].iloc[0]) if not gp_row.empty else np.nan
        tp = int(gp_row['total_points'].iloc[0]) if not gp_row.empty else np.nan
        severity_first = (float('inf') if (gp == 0 and max_first > 0) else (max_first / gp) ) if not np.isnan(gp) else np.nan
        metric_faults_list.append({
            'cmdb_id': cmdb,
            'kpi_name': kpi,
            'total_points': tp,
            'global_p95': gp,
            'number_of_faults': number_of_faults,
            'longest_fault_length': longest_fault_length,
            'first_fault_start_ts': first_start,
            'first_fault_end_ts': first_end,
            'max_value_first_fault': max_first,
            'severity_ratio_first_fault': severity_first
        })

metric_faults_df = pd.DataFrame(metric_faults_list)

# Per-component earliest_metric_fault and severity_ratio_metric
if not metric_faults_df.empty:
    mf = metric_faults_df.copy()
    comp_metric = mf.groupby('cmdb_id').agg(
        earliest_metric_fault_ts=('first_fault_start_ts','min'),
        severity_ratio_metric=('severity_ratio_first_fault','max')
    ).reset_index()
else:
    comp_metric = pd.DataFrame(columns=['cmdb_id','earliest_metric_fault_ts','severity_ratio_metric'])

# -------------------------
# Trace: compute global_p95, detect anomalies in incident window and aggregate per component
# -------------------------
trace_global = (
    df_trace
    .groupby(['cmdb_id','trace_name'], as_index=False)
    .agg(total_points=('value','count'),
         global_p95=('value', lambda s: float(s.quantile(0.95))))
)

df_trace_inc = df_trace[(df_trace['timestamp'] >= start_epoch) & (df_trace['timestamp'] <= end_epoch)].copy()
df_trace_inc = pd.merge(df_trace_inc, trace_global[['cmdb_id','trace_name','global_p95','total_points']], on=['cmdb_id','trace_name'], how='left')

df_trace_anom = df_trace_inc[df_trace_inc['value'] > df_trace_inc['global_p95']].copy()

trace_anom_list = []
if not df_trace_anom.empty:
    tg = df_trace_anom.groupby(['cmdb_id','trace_name'])
    for (cmdb, tname), grp in tg:
        earliest_ts = int(grp['timestamp'].min())
        max_val = float(grp['value'].max())
        gp = float(trace_global[(trace_global['cmdb_id']==cmdb)&(trace_global['trace_name']==tname)]['global_p95'].iloc[0])
        severity = float('inf') if (gp == 0 and max_val > 0) else (max_val / gp) if gp != 0 else np.nan
        trace_anom_list.append({
            'cmdb_id': cmdb,
            'trace_name': tname,
            'earliest_anomaly_ts': earliest_ts,
            'max_anomaly_value': max_val,
            'severity_ratio': severity
        })

trace_anom_df = pd.DataFrame(trace_anom_list)

# Per-component earliest_trace_anomaly and severity_ratio_trace
if not trace_anom_df.empty:
    ct = trace_anom_df.copy()
    comp_trace = ct.groupby('cmdb_id').agg(
        earliest_trace_anomaly_ts=('earliest_anomaly_ts','min'),
        severity_ratio_trace=('severity_ratio','max')
    ).reset_index()
else:
    comp_trace = pd.DataFrame(columns=['cmdb_id','earliest_trace_anomaly_ts','severity_ratio_trace'])

# -------------------------
# Log: compute global_p95, detect anomalies in incident window and aggregate per component
# -------------------------
log_global = (
    df_log
    .groupby(['cmdb_id','log_name'], as_index=False)
    .agg(total_points=('value','count'),
         global_p95=('value', lambda s: float(s.quantile(0.95))))
)

df_log_inc = df_log[(df_log['timestamp'] >= start_epoch) & (df_log['timestamp'] <= end_epoch)].copy()
df_log_inc = pd.merge(df_log_inc, log_global[['cmdb_id','log_name','global_p95','total_points']], on=['cmdb_id','log_name'], how='left')
df_log_anom = df_log_inc[df_log_inc['value'] > df_log_inc['global_p95']].copy()

log_anom_list = []
if not df_log_anom.empty:
    lg = df_log_anom.groupby(['cmdb_id','log_name'])
    for (cmdb, lname), grp in lg:
        earliest_ts = int(grp['timestamp'].min())
        max_val = float(grp['value'].max())
        gp = float(log_global[(log_global['cmdb_id']==cmdb)&(log_global['log_name']==lname)]['global_p95'].iloc[0])
        severity = float('inf') if (gp == 0 and max_val > 0) else (max_val / gp) if gp != 0 else np.nan
        log_anom_list.append({
            'cmdb_id': cmdb,
            'log_name': lname,
            'earliest_anomaly_ts': earliest_ts,
            'max_anomaly_value': max_val,
            'severity_ratio': severity
        })

log_anom_df = pd.DataFrame(log_anom_list)

# Per-component earliest_log_anomaly and severity_ratio_log
if not log_anom_df.empty:
    cl = log_anom_df.copy()
    comp_log = cl.groupby('cmdb_id').agg(
        earliest_log_anomaly_ts=('earliest_anomaly_ts','min'),
        severity_ratio_log=('severity_ratio','max')
    ).reset_index()
else:
    comp_log = pd.DataFrame(columns=['cmdb_id','earliest_log_anomaly_ts','severity_ratio_log'])

# -------------------------
# Build correlation table for candidates
# -------------------------
# Start with candidates DataFrame
corr = pd.DataFrame({'component': candidates})

# Merge metric, trace, log component summaries
corr = corr.merge(comp_metric.rename(columns={
    'cmdb_id':'component',
    'earliest_metric_fault_ts':'earliest_metric_fault_ts',
    'severity_ratio_metric':'severity_ratio_metric'
}), on='component', how='left')

corr = corr.merge(comp_trace.rename(columns={
    'cmdb_id':'component',
    'earliest_trace_anomaly_ts':'earliest_trace_anomaly_ts',
    'severity_ratio_trace':'severity_ratio_trace'
}), on='component', how='left')

corr = corr.merge(comp_log.rename(columns={
    'cmdb_id':'component',
    'earliest_log_anomaly_ts':'earliest_log_anomaly_ts',
    'severity_ratio_log':'severity_ratio_log'
}), on='component', how='left')

# Convert epoch timestamps to Asia/Shanghai strings or keep null
def epoch_to_local_str_nullable(v):
    if pd.isna(v):
        return None
    return pd.to_datetime(int(v), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S')

corr['earliest_metric_fault'] = corr['earliest_metric_fault_ts'].apply(epoch_to_local_str_nullable)
corr['earliest_trace_anomaly'] = corr['earliest_trace_anomaly_ts'].apply(epoch_to_local_str_nullable)
corr['earliest_log_anomaly'] = corr['earliest_log_anomaly_ts'].apply(epoch_to_local_str_nullable)

# Keep requested columns and drop intermediate ts columns
corr_table = corr[[
    'component',
    'earliest_metric_fault','severity_ratio_metric',
    'earliest_trace_anomaly','severity_ratio_trace',
    'earliest_log_anomaly','severity_ratio_log'
]]

# -------------------------
# Find earliest non-null timestamp among the three timestamp columns across rows
# -------------------------
# Prepare a helper DataFrame with component and each timestamp as epoch for comparison
def to_epoch_or_nan(s):
    if pd.isna(s):
        return np.nan
    return int(s)

# Build a DataFrame with epoch timestamps for comparison
comp_times = corr[['component','earliest_metric_fault_ts','earliest_trace_anomaly_ts','earliest_log_anomaly_ts']].copy()
# Melt to long form
melt = comp_times.melt(id_vars=['component'], value_vars=['earliest_metric_fault_ts','earliest_trace_anomaly_ts','earliest_log_anomaly_ts'],
                       var_name='source', value_name='ts')
melt = melt.dropna(subset=['ts'])
if not melt.empty:
    earliest_row = melt.sort_values('ts').iloc[0]
    earliest_component = earliest_row['component']
    earliest_ts_epoch = int(earliest_row['ts'])
    earliest_ts_local = epoch_to_local_str_nullable(earliest_ts_epoch)
    earliest_nonnull = pd.DataFrame([{
        'component': earliest_component,
        'earliest_cross_modal_timestamp': earliest_ts_local
    }])
else:
    earliest_nonnull = pd.DataFrame([{'component': None, 'earliest_cross_modal_timestamp': None}])

# Final compact outputs: corr_table (rows for 8 candidates) and earliest_nonnull single-row DataFrame
corr_table, earliest_nonnull

```
Out[7]:
```
Summary correlation for the candidate components (compact):

- MG01
  - earliest_metric_fault: 2021-03-23 18:30:00
  - severity_ratio_metric: inf (global_p95 == 0 → positive spike)
  - earliest_trace_anomaly: 2021-03-23 18:34:00
  - severity_ratio_trace: 3.752
  - earliest_log_anomaly: null
  - severity_ratio_log: null

- MG02
  - earliest_metric_fault: 2021-03-23 18:30:00
  - severity_ratio_metric: inf
  - earliest_trace_anomaly: 2021-03-23 18:33:00
  - severity_ratio_trace: 2.863
  - earliest_log_anomaly: null
  - severity_ratio_log: null

- Tomcat01
  - earliest_metric_fault: 2021-03-23 18:30:00
  - severity_ratio_metric: 4.000
  - earliest_trace_anomaly: 2021-03-23 18:37:00
  - severity_ratio_trace: 3.752
  - earliest_log_anomaly: 2021-03-23 18:50:00
  - severity_ratio_log: 1.152

- Tomcat02
  - earliest_metric_fault: 2021-03-23 18:30:00
  - severity_ratio_metric: 3.333
  - earliest_trace_anomaly: 2021-03-23 18:34:00
  - severity_ratio_trace: 1.640
  - earliest_log_anomaly: 2021-03-23 18:34:00
  - severity_ratio_log: 1.048

- Tomcat03
  - earliest_metric_fault: 2021-03-23 18:30:00
  - severity_ratio_metric: 3.105
  - earliest_trace_anomaly: 2021-03-23 18:34:00
  - severity_ratio_trace: 1.339
  - earliest_log_anomaly: 2021-03-23 18:39:00
  - severity_ratio_log: 1.159

- Tomcat04
  - earliest_metric_fault: 2021-03-23 18:30:00
  - severity_ratio_metric: 3.316
  - earliest_trace_anomaly: 2021-03-23 18:35:00
  - severity_ratio_trace: 1.348
  - earliest_log_anomaly: 2021-03-23 18:35:00
  - severity_ratio_log: 1.070

- Mysql02
  - earliest_metric_fault: 2021-03-23 18:30:00
  - severity_ratio_metric: inf
  - earliest_trace_anomaly: null
  - severity_ratio_trace: null
  - earliest_log_anomaly: 2021-03-23 18:59:00
  - severity_ratio_log: 1.009

- Redis02
  - earliest_metric_fault: 2021-03-23 18:31:00
  - severity_ratio_metric: 3.333
  - earliest_trace_anomaly: null
  - severity_ratio_trace: null
  - earliest_log_anomaly: null
  - severity_ratio_log: null

Earliest cross-modal anomaly (single earliest non-null timestamp across metric/trace/log):
- Component: MG01
- Timestamp: 2021-03-23 18:30:00 (UTC+8)

Interpretation / guidance:
- Metric faults start earliest (18:30) across many candidates; MG01 is the earliest cross-modal hit and shows an infinite metric severity_ratio (global_p95 == 0 with a positive spike), making it a high-priority candidate for root-cause investigation.
- MG02 and multiple Tomcat hosts also show early metric faults at 18:30 and notable trace anomalies shortly after (~18:33–18:37). Logs show later, smaller anomalies for several Tomcats and Mysql02.
- Use MG01 first as the top lead, then inspect MG02 and the Tomcat hosts and their trace paths around 18:30–18:35.

The original code execution output of IPython Kernel is also provided below for reference:

(  component earliest_metric_fault  severity_ratio_metric earliest_trace_anomaly  severity_ratio_trace earliest_log_anomaly  severity_ratio_log
0      MG01   2021-03-23 18:30:00                    inf    2021-03-23 18:34:00              3.752298                 None                 NaN
1      MG02   2021-03-23 18:30:00                    inf    2021-03-23 18:33:00              2.862718                 None                 NaN
2  Tomcat01   2021-03-23 18:30:00               4.000000    2021-03-23 18:37:00              3.752298  2021-03-23 18:50:00            1.151737
3  Tomcat02   2021-03-23 18:30:00               3.333333    2021-03-23 18:34:00              1.640118  2021-03-23 18:34:00            1.047788
4  Tomcat03   2021-03-23 18:30:00               3.104984    2021-03-23 18:34:00              1.338944  2021-03-23 18:39:00            1.159420
5  Tomcat04   2021-03-23 18:30:00               3.316444    2021-03-23 18:35:00              1.347935  2021-03-23 18:35:00            1.069519
6   Mysql02   2021-03-23 18:30:00                    inf                   None                   NaN  2021-03-23 18:59:00            1.009394
7   Redis02   2021-03-23 18:31:00               3.333333                   None                   NaN                 None                 NaN,   component earliest_cross_modal_timestamp
0      MG01            2021-03-23 18:30:00)```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

# reuse df_metric and metric_global if available, otherwise load/recompute
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

try:
    metric_global
except NameError:
    metric_global = (
        df_metric
        .groupby(['cmdb_id','kpi_name'], as_index=False)
        .agg(
            total_points=('value','count'),
            global_p95=('value', lambda s: float(s.quantile(0.95)))
        )
    )

tz = pytz.timezone('Asia/Shanghai')

# Incident window (Asia/Shanghai) -> UTC epoch seconds
start_local = pd.Timestamp('2021-03-23 18:30:00', tz=tz)
end_local   = pd.Timestamp('2021-03-23 19:00:00', tz=tz)
start_epoch = int(start_local.tz_convert('UTC').timestamp())
end_epoch   = int(end_local.tz_convert('UTC').timestamp())

# Filter incident-window metric rows and MG01 rows
df_metric_inc = df_metric[(df_metric['timestamp'] >= start_epoch) & (df_metric['timestamp'] <= end_epoch)].copy()
df_mg_inc = df_metric_inc[df_metric_inc['cmdb_id'] == 'MG01'].copy()

# If there are no MG01 incident rows, return empty results
if df_mg_inc.empty:
    final_table = pd.DataFrame(columns=[
        'kpi_name','total_points','global_p95','incident_points',
        'anomaly_count','earliest_anomaly_timestamp','max_anomaly_value','severity_ratio'
    ])
    summary = "MG01 has no KPI anomaly points in the incident window."
else:
    # Merge global p95 and total_points for MG01 KPIs
    mg_metric_global = metric_global[metric_global['cmdb_id']=='MG01'][['kpi_name','total_points','global_p95']].copy()
    df_mg_inc = pd.merge(df_mg_inc, mg_metric_global, on='kpi_name', how='left')

    # Incident points per KPI
    incident_counts = df_mg_inc.groupby('kpi_name', as_index=False).agg(incident_points=('value','count'))

    # Identify anomaly points (value > global_p95)
    df_mg_anom = df_mg_inc[df_mg_inc['value'] > df_mg_inc['global_p95']].copy()

    if df_mg_anom.empty:
        final_table = pd.DataFrame(columns=[
            'kpi_name','total_points','global_p95','incident_points',
            'anomaly_count','earliest_anomaly_timestamp','max_anomaly_value','severity_ratio'
        ])
        summary = "MG01 has no KPI anomaly points in the incident window (no values > global_p95)."
    else:
        # Aggregate anomaly info per KPI
        anom_agg = (
            df_mg_anom
            .groupby('kpi_name', as_index=False)
            .agg(
                anomaly_count=('value','count'),
                earliest_anomaly_timestamp_epoch=('timestamp','min'),
                max_anomaly_value=('value','max')
            )
        )

        # Compute fault segments per KPI to find longest fault length and first fault start
        fault_rows = []
        grp = df_mg_anom.sort_values(['kpi_name','timestamp']).groupby('kpi_name')
        for kpi, g in grp:
            times = g['timestamp'].values
            vals = g['value'].values
            # compute consecutive segments where diff == 60
            diffs = np.diff(times, prepend=times[0])
            seg_ids = (diffs != 60).cumsum()
            seg_df = pd.DataFrame({'ts': times, 'val': vals, 'seg': seg_ids})
            seg_summary = seg_df.groupby('seg').agg(
                seg_start=('ts','min'),
                seg_end=('ts','max'),
                seg_len_points=('ts','count'),
                seg_max_value=('val','max')
            ).reset_index(drop=True)
            # longest segment length (points)
            longest_len = int(seg_summary['seg_len_points'].max())
            # first fault start timestamp (earliest segment)
            first_start_epoch = int(seg_summary.loc[seg_summary['seg_start'].idxmin(),'seg_start'])
            fault_rows.append({
                'kpi_name': kpi,
                'longest_fault_length': longest_len,
                'first_fault_start_epoch': first_start_epoch
            })

        fault_df = pd.DataFrame(fault_rows)

        # Combine aggregates
        combined = anom_agg.merge(incident_counts, on='kpi_name', how='left')
        combined = combined.merge(mg_metric_global, on='kpi_name', how='left')
        combined = combined.merge(fault_df, on='kpi_name', how='left')

        # Compute severity_ratio (handle global_p95 == 0)
        def compute_sev(row):
            gp = row['global_p95']
            mv = row['max_anomaly_value']
            if pd.isna(gp):
                return np.nan
            if gp == 0:
                return float('inf') if mv > 0 else np.nan
            return mv / gp

        combined['severity_ratio'] = combined.apply(compute_sev, axis=1)

        # Convert earliest_anomaly_timestamp and first_fault_start to Asia/Shanghai readable strings
        def epoch_to_local_str(v):
            if pd.isna(v):
                return None
            return pd.to_datetime(int(v), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S')

        combined['earliest_anomaly_timestamp'] = combined['earliest_anomaly_timestamp_epoch'].apply(epoch_to_local_str)
        combined['first_fault_start_timestamp'] = combined['first_fault_start_epoch'].apply(epoch_to_local_str)

        # Select requested columns and sort by earliest_anomaly_timestamp ascending, limit top 50
        final_table = combined[[
            'kpi_name','total_points','global_p95','incident_points',
            'anomaly_count','earliest_anomaly_timestamp','max_anomaly_value','severity_ratio',
            'longest_fault_length','first_fault_start_timestamp'
        ]].sort_values(by='earliest_anomaly_timestamp', ascending=True).head(50)

        # Build single-line summary about any fault longer than 1 minute
        long_faults = final_table[final_table['longest_fault_length'] > 1]
        if long_faults.empty:
            summary = "MG01 has no KPI with a fault sub-series longer than 1 minute."
        else:
            # choose the earliest first_fault_start_timestamp among those
            lf = long_faults.copy()
            lf['first_fault_start_epoch_int'] = pd.to_datetime(lf['first_fault_start_timestamp']).astype('int64') // 10**9
            lf = lf.sort_values('first_fault_start_epoch_int')
            kpi_name = lf.iloc[0]['kpi_name']
            first_start = lf.iloc[0]['first_fault_start_timestamp']
            summary = f"MG01 has at least one KPI with fault >1 minute: {kpi_name}, first_fault_start_timestamp={first_start}."

# Present compact outputs: final_table (top up to 50 rows) and summary string
final_table.head(50), summary

```
Out[8]:
```
```python
# Summarize MG01 KPI anomaly findings in plain English using existing results (final_table and summary)
try:
    final_table
    summary
except NameError:
    raise RuntimeError("Required variables (final_table, summary) not found in session. Please run the previous analysis steps.")

# compact stats
mg_kpi_count = int(final_table.shape[0])
earliest_anom = final_table['earliest_anomaly_timestamp'].min() if mg_kpi_count>0 else None
# top severity row (handle possible inf)
top_sev_row = None
if mg_kpi_count>0:
    top_sev_row = final_table.sort_values(by='severity_ratio', ascending=False).iloc[0]

top_sev_info = ""
if top_sev_row is not None:
    k = top_sev_row['kpi_name']
    s = top_sev_row['severity_ratio']
    t = top_sev_row['earliest_anomaly_timestamp']
    s_str = "inf" if (pd.isna(s) == False and (s == float('inf') or str(s) == 'inf')) else f"{float(s):.3g}"
    top_sev_info = f"Highest severity KPI: {k} (severity_ratio={s_str}) first anomaly at {t}."

summary_text = (
    f"MG01 metric anomaly summary (incident window 2021-03-23 18:30–19:00 UTC+8):\n"
    f"- Number of MG01 KPIs with anomaly points: {mg_kpi_count}\n"
    f"- Earliest anomaly among these KPIs: {earliest_anom}\n"
    f"- {top_sev_info}\n"
    f"- Fault-length check: {summary}\n"
    "- Note: Most detected faults are single-minute spikes (longest_fault_length = 1) except where noted above."
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(                                             kpi_name  total_points    global_p95  incident_points  anomaly_count earliest_anomaly_timestamp  max_anomaly_value  severity_ratio  longest_fault_length first_fault_start_timestamp
34    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead            15  2.666769e-01               15              1        2021-03-23 18:30:00       2.667000e-01        1.000087                     1         2021-03-23 18:30:00
36    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWTps            15  1.863300e+00               15              1        2021-03-23 18:30:00       1.933300e+00        1.037568                     1         2021-03-23 18:30:00
25  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...            15  2.300000e-01               15              1        2021-03-23 18:32:00       3.000000e-01        1.304348                     1         2021-03-23 18:32:00
20  OSLinux-OSLinux_FILESYSTEM_-tomcat_FSAvailable...             3  2.570700e+03                3              1        2021-03-23 18:32:00       2.579000e+03        1.003229                     1         2021-03-23 18:32:00
3            JVM-Threads_7779_JVM_ThreadCount_Threads            31  7.550000e+01               31              2        2021-03-23 18:33:00       7.600000e+01        1.006623                     1         2021-03-23 18:33:00
..                                                ...           ...           ...              ...            ...                        ...                ...             ...                   ...                         ...
19  OSLinux-OSLinux_FILESYSTEM_----tomcat_FSInodeU...            15  2.897765e-01               15              1        2021-03-23 18:59:00       2.899480e-01        1.000592                     1         2021-03-23 18:59:00
44        OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn            15  2.743410e+09               15              1        2021-03-23 18:59:00       2.743463e+09        1.000020                     1         2021-03-23 18:59:00
17     OSLinux-OSLinux_FILESYSTEM_----home_FSCapacity            15  2.721463e+00               15              1        2021-03-23 18:59:00       2.721492e+00        1.000011                     1         2021-03-23 18:59:00
45       OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut            15  2.574979e+09               15              1        2021-03-23 18:59:00       2.575029e+09        1.000019                     1         2021-03-23 18:59:00
41    OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum            21  2.840000e+03               21              1        2021-03-23 19:00:00       2.858000e+03        1.006338                     1         2021-03-23 19:00:00

[47 rows x 10 columns], 'MG01 has at least one KPI with fault >1 minute: OSLinux-CPU_CPU_CPUidleutil, first_fault_start_timestamp=2021-03-23 18:47:00.')

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz

# reuse df_trace if available
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

tz = pytz.timezone('Asia/Shanghai')

# Incident full window (for earlier steps) and the specific filter window
incident_start_local = pd.Timestamp('2021-03-23 18:30:00', tz=tz)
incident_end_local   = pd.Timestamp('2021-03-23 19:00:00', tz=tz)
incident_start_epoch = int(incident_start_local.tz_convert('UTC').timestamp())
incident_end_epoch   = int(incident_end_local.tz_convert('UTC').timestamp())

# Target earliest-anomaly filter window: 18:33:00 to 18:35:00 inclusive (UTC+8)
filter_start_local = pd.Timestamp('2021-03-23 18:33:00', tz=tz)
filter_end_local   = pd.Timestamp('2021-03-23 18:35:00', tz=tz)
filter_start_epoch = int(filter_start_local.tz_convert('UTC').timestamp())
filter_end_epoch   = int(filter_end_local.tz_convert('UTC').timestamp())

# 1) Compute global_p95 per (cmdb_id, trace_name) using full series (rule 9)
trace_global = (
    df_trace
    .groupby(['cmdb_id','trace_name'], as_index=False)
    .agg(
        total_points=('value','count'),
        global_p95=('value', lambda s: float(s.quantile(0.95)))
    )
)

# 2) Filter trace rows to the incident window
df_trace_inc = df_trace[(df_trace['timestamp'] >= incident_start_epoch) & (df_trace['timestamp'] <= incident_end_epoch)].copy()

# 3) Merge global_p95 into incident rows
df_trace_inc = pd.merge(df_trace_inc, trace_global[['cmdb_id','trace_name','global_p95','total_points']], on=['cmdb_id','trace_name'], how='left')

# 4) Identify anomaly rows in incident window (value > global_p95)
df_trace_anom = df_trace_inc[df_trace_inc['value'] > df_trace_inc['global_p95']].copy()

# 5) Aggregate per group: incident_points, earliest_anomaly_timestamp (epoch), max_anomaly_value
if not df_trace_anom.empty:
    agg = (
        df_trace_anom
        .groupby(['cmdb_id','trace_name'], as_index=False)
        .agg(
            incident_points=('value','count'),
            earliest_anomaly_timestamp_epoch=('timestamp','min'),
            max_anomaly_value=('value','max')
        )
    )
    # merge global_p95
    agg = pd.merge(agg, trace_global[['cmdb_id','trace_name','global_p95']], on=['cmdb_id','trace_name'], how='left')
    # compute severity_ratio (handle global_p95 == 0)
    def compute_sev(row):
        gp = row['global_p95']
        mv = row['max_anomaly_value']
        if pd.isna(gp):
            return np.nan
        if gp == 0:
            return float('inf') if mv > 0 else np.nan
        return mv / gp
    agg['severity_ratio'] = agg.apply(compute_sev, axis=1)
else:
    agg = pd.DataFrame(columns=[
        'cmdb_id','trace_name','incident_points','earliest_anomaly_timestamp_epoch','max_anomaly_value','global_p95','severity_ratio'
    ])

# 6) Filter groups whose earliest_anomaly_timestamp is within [18:33, 18:35] UTC+8
if not agg.empty:
    mask = (agg['earliest_anomaly_timestamp_epoch'] >= filter_start_epoch) & (agg['earliest_anomaly_timestamp_epoch'] <= filter_end_epoch)
    filtered = agg[mask].copy()
else:
    filtered = agg.copy()

# 7) Prepare display columns and convert epoch to UTC+8 string
def epoch_to_local_str(v):
    if pd.isna(v):
        return None
    return pd.to_datetime(int(v), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S')

if not filtered.empty:
    filtered['earliest_anomaly_timestamp'] = filtered['earliest_anomaly_timestamp_epoch'].apply(epoch_to_local_str)
    final_df = filtered[[
        'cmdb_id','trace_name','earliest_anomaly_timestamp','max_anomaly_value','global_p95','severity_ratio','incident_points'
    ]].sort_values(by='earliest_anomaly_timestamp', ascending=True).head(50).reset_index(drop=True)
    unique_cmdb_ids = set(final_df['cmdb_id'].unique())
else:
    final_df = filtered[[
        'cmdb_id','trace_name','earliest_anomaly_timestamp_epoch','max_anomaly_value','global_p95','severity_ratio','incident_points'
    ]].head(0)
    unique_cmdb_ids = set()

# Display compact results
final_df, unique_cmdb_ids

```
Out[9]:
```
Summary of trace-group anomalies between 2021-03-23 18:33:00 and 18:35:00 (UTC+8):

- Total trace groups found in this window: 39 groups.
- Earliest anomalies occur at 18:33:00 (UTC+8): examples include
  - dockerA2 — trace.to_MG02.row_count (max=62, global_p95=59, severity≈1.051, incident_points=2)
  - MG02 — trace.from_dockerA2.row_count (same spike pair)
- A cluster of anomalies follows at 18:34:00 involving MG01-related flows and others, e.g.:
  - IG02 — trace.self.row_count (max=1126 vs p95=1077, severity≈1.045)
  - dockerB1 — trace.to_MG01.row_count (max=252 vs p95=217.5, severity≈1.159)
  - dockerB2 — trace.from_MG01.row_count (max=77 vs p95=72, severity≈1.069)
- At 18:35:00 there are further anomalies (examples):
  - Tomcat04 — trace.from_IG02.row_count (max≈148, severity≈1.031)
  - dockerA1 — trace.to_MG01.duration_p95 (max≈2.773, global_p95≈1.395, severity≈1.988) — one of the larger relative spikes in this window.
- Severity patterns: mostly modest increases above p95 (severity ratios ≈1.03–1.16); a few stronger relative spikes (up to ≈1.99).
- Incident point counts for these groups are small and uniform (typically incident_points = 2).
- Unique components observed in the filtered set: {'MG01','MG02','dockerA1','dockerA2','dockerB1','dockerB2','IG02','Tomcat02','Tomcat03','Tomcat04'}.

Interpretation / next steps:
- There is an early spike pair at 18:33 between dockerA2 and MG02 — investigate that path first.
- A broader cluster involving MG01 (dockerB1/dockerB2/dockerA1) appears at ~18:34–18:35; dockerA1 → MG01 duration_p95 shows one of the largest relative anomalies and merits inspection.
- Overall, anomalies in this 3-minute window are short-lived and mostly modest; focus tracing on the listed components and the specific trace flows around 18:33–18:35 UTC+8.

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                       trace_name earliest_anomaly_timestamp  max_anomaly_value   global_p95  severity_ratio  incident_points
0   dockerA2          trace.to_MG02.row_count        2021-03-23 18:33:00           62.00000    59.000000        1.050847                2
1       MG02    trace.from_dockerA2.row_count        2021-03-23 18:33:00           62.00000    59.000000        1.050847                2
2       IG02             trace.self.row_count        2021-03-23 18:34:00         1126.00000  1077.000000        1.045497                2
3   dockerB2        trace.from_MG01.row_count        2021-03-23 18:34:00           77.00000    72.000000        1.069444                2
4   dockerB1          trace.to_MG01.row_count        2021-03-23 18:34:00          252.00000   217.500000        1.158621                2
..       ...                              ...                        ...                ...          ...             ...              ...
34  Tomcat04        trace.from_IG02.row_count        2021-03-23 18:35:00          148.00000   143.500000        1.031359                2
35      MG02   trace.to_dockerA1.duration_p95        2021-03-23 18:35:00            1.35690     1.309145        1.036478                2
36      MG02  trace.to_dockerA1.duration_mean        2021-03-23 18:35:00            0.33300     0.298611        1.115164                2
37  dockerA1          trace.to_MG01.row_count        2021-03-23 18:35:00           29.00000    21.550000        1.345708                2
38  dockerA1       trace.to_MG01.duration_p95        2021-03-23 18:35:00            2.77325     1.395270        1.987608                2

[39 rows x 7 columns], {'MG01', 'dockerB1', 'dockerB2', 'Tomcat04', 'dockerA1', 'IG02', 'Tomcat03', 'Tomcat02', 'dockerA2', 'MG02'})

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np
import pytz

# Reuse df_metric if available, otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

tz = pytz.timezone('Asia/Shanghai')

# Candidate component
comp = 'MG02'

# Incident window (Asia/Shanghai) -> UTC epoch seconds
start_local = pd.Timestamp('2021-03-23 18:30:00', tz=tz)
end_local   = pd.Timestamp('2021-03-23 19:00:00', tz=tz)
start_epoch = int(start_local.tz_convert('UTC').timestamp())
end_epoch   = int(end_local.tz_convert('UTC').timestamp())

# 1) Compute global_p95 and total_points per MG02 kpi_name using the full MG02 series (rule 9)
mg02_all = df_metric[df_metric['cmdb_id'] == comp].copy()
if mg02_all.empty:
    final_table = pd.DataFrame(columns=[
        'kpi_name','total_points','global_p95','incident_points',
        'anomaly_count','earliest_anomaly_timestamp','max_anomaly_value',
        'severity_ratio','longest_fault_length','first_fault_start_timestamp'
    ])
    indicator = f"No metric data for {comp} in metric.csv."
else:
    mg02_global = (
        mg02_all
        .groupby('kpi_name', as_index=False)
        .agg(
            total_points=('value','count'),
            global_p95=('value', lambda s: float(s.quantile(0.95)))
        )
    )

    # 2) Filter MG02 rows to incident window
    mg02_inc = mg02_all[(mg02_all['timestamp'] >= start_epoch) & (mg02_all['timestamp'] <= end_epoch)].copy()
    if mg2 := mg02_inc.empty:
        # no incident rows
        final_table = pd.DataFrame(columns=[
            'kpi_name','total_points','global_p95','incident_points',
            'anomaly_count','earliest_anomaly_timestamp','max_anomaly_value',
            'severity_ratio','longest_fault_length','first_fault_start_timestamp'
        ])
        indicator = f"{comp} has no incident-window metric rows."
    else:
        # incident_points per KPI
        incident_counts = mg02_inc.groupby('kpi_name', as_index=False).agg(incident_points=('value','count'))

        # merge global thresholds into incident rows to detect anomalies
        mg02_inc = pd.merge(mg02_inc, mg02_global, on='kpi_name', how='left')

        # anomaly points: value > global_p95
        mg02_anom = mg02_inc[mg02_inc['value'] > mg02_inc['global_p95']].copy()

        if mg02_anom.empty:
            final_table = pd.DataFrame(columns=[
                'kpi_name','total_points','global_p95','incident_points',
                'anomaly_count','earliest_anomaly_timestamp','max_anomaly_value',
                'severity_ratio','longest_fault_length','first_fault_start_timestamp'
            ])
            indicator = f"{comp} has no KPI anomaly points in the incident window (no values > global_p95)."
        else:
            # Aggregate anomaly info per KPI
            anom_agg = (
                mg02_anom
                .groupby('kpi_name', as_index=False)
                .agg(
                    anomaly_count=('value','count'),
                    earliest_anomaly_timestamp_epoch=('timestamp','min'),
                    max_anomaly_value=('value','max')
                )
            )

            # Compute fault segments per KPI to find longest fault length and first fault start
            fault_rows = []
            grp = mg02_anom.sort_values(['kpi_name','timestamp']).groupby('kpi_name')
            for kpi, g in grp:
                times = g['timestamp'].values
                vals = g['value'].values
                # compute consecutive segments where diff == 60 seconds
                diffs = np.diff(times, prepend=times[0])
                seg_ids = (diffs != 60).cumsum()
                seg_df = pd.DataFrame({'ts': times, 'val': vals, 'seg': seg_ids})
                seg_summary = seg_df.groupby('seg').agg(
                    seg_start=('ts','min'),
                    seg_end=('ts','max'),
                    seg_len_points=('ts','count'),
                    seg_max_value=('val','max')
                ).reset_index(drop=True)
                longest_len = int(seg_summary['seg_len_points'].max())
                # first fault start epoch (earliest seg_start)
                first_start_epoch = int(seg_summary['seg_start'].min())
                fault_rows.append({
                    'kpi_name': kpi,
                    'longest_fault_length': longest_len,
                    'first_fault_start_epoch': first_start_epoch
                })

            fault_df = pd.DataFrame(fault_rows)

            # Combine aggregates: total_points, global_p95, incident_points, anomaly aggregates, fault info
            combined = anom_agg.merge(incident_counts, on='kpi_name', how='left')
            combined = combined.merge(mg02_global, on='kpi_name', how='left')
            combined = combined.merge(fault_df, on='kpi_name', how='left')

            # Compute severity_ratio handling global_p95 == 0
            def compute_sev(row):
                gp = row['global_p95']
                mv = row['max_anomaly_value']
                if pd.isna(gp):
                    return np.nan
                if gp == 0:
                    return float('inf') if mv > 0 else np.nan
                return mv / gp

            combined['severity_ratio'] = combined.apply(compute_sev, axis=1)

            # Convert epoch timestamps to Asia/Shanghai readable strings
            def epoch_to_local_str(v):
                if pd.isna(v):
                    return None
                return pd.to_datetime(int(v), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S')

            combined['earliest_anomaly_timestamp'] = combined['earliest_anomaly_timestamp_epoch'].apply(epoch_to_local_str)
            combined['first_fault_start_timestamp'] = combined['first_fault_start_epoch'].apply(epoch_to_local_str)

            # Select requested columns, sort by earliest_anomaly_timestamp ascending, limit top 50
            final_table = combined[[
                'kpi_name','total_points','global_p95','incident_points',
                'anomaly_count','earliest_anomaly_timestamp','max_anomaly_value',
                'severity_ratio','longest_fault_length','first_fault_start_timestamp'
            ]].sort_values(by='earliest_anomaly_timestamp', ascending=True).head(50).reset_index(drop=True)

            # Indicator: any KPI with longest_fault_length > 1 ?
            long_faults = final_table[final_table['longest_fault_length'] > 1]
            if long_faults.empty:
                indicator = f"{comp} has no KPI with longest_fault_length > 1."
            else:
                # choose the KPI with earliest first_fault_start_timestamp among these
                lf = long_faults.copy()
                # convert timestamp string back to epoch for sorting safely
                lf['first_start_epoch'] = pd.to_datetime(lf['first_fault_start_timestamp']).astype('int64') // 10**9
                lf = lf.sort_values('first_start_epoch')
                kpi_name = lf.iloc[0]['kpi_name']
                first_start = lf.iloc[0]['first_fault_start_timestamp']
                indicator = f"{comp} has KPI with longest_fault_length > 1: {kpi_name}, first_fault_start_timestamp={first_start}."

# Display compact outputs: final_table and indicator
final_table, indicator

```
Out[10]:
```
# Prepare a concise plain-English summary of MG02 metric anomalies using existing results
try:
    final_table
    indicator
except NameError:
    raise RuntimeError("Required variables (final_table, indicator) not found in session. Please run the MG02 analysis code first.")

# Build top-5 example lines
examples = []
if final_table.shape[0] > 0:
    for r in final_table.head(5).itertuples(index=False):
        # columns: kpi_name,total_points,global_p95,incident_points,anomaly_count,earliest_anomaly_timestamp,
        # max_anomaly_value,severity_ratio,longest_fault_length,first_fault_start_timestamp
        sev = r.severity_ratio
        sev_str = "inf" if (pd.notna(sev) and (sev == float('inf') or str(sev) == 'inf')) else (f"{float(sev):.3g}" if pd.notna(sev) else "null")
        examples.append(f"{r.kpi_name} | earliest={r.earliest_anomaly_timestamp} | max={float(r.max_anomaly_value):.6g} | severity={sev_str}")
else:
    examples.append("No MG02 KPI anomalies in the incident window.")

summary_text = (
    f"MG02 metric anomaly summary (2021-03-23 18:30–19:00 UTC+8):\n"
    f"- Number of MG02 KPIs with anomaly points: {int(final_table.shape[0])}\n"
    f"- Earliest anomaly timestamp among these KPIs: {final_table['earliest_anomaly_timestamp'].min() if final_table.shape[0]>0 else 'None'}\n"
    f"- Fault-length check: {indicator}\n"
    f"- Example anomalous KPIs (top 5 by earliest time):\n  " + "\n  ".join(examples) + "\n\n"
    "Notes: severity_ratio = max_anomaly_value / global_p95 (inf if global_p95==0). Most detected faults are single-minute spikes (longest_fault_length = 1) per the indicator above."
)

summary_text

The original code execution output of IPython Kernel is also provided below for reference:

(                                             kpi_name  total_points    global_p95  incident_points  anomaly_count earliest_anomaly_timestamp  max_anomaly_value  severity_ratio  longest_fault_length first_fault_start_timestamp
0     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead            15  6.120000e+00               15              1        2021-03-23 18:30:00       6.400000e+00        1.045752                     1         2021-03-23 18:30:00
1   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRea...             7  9.552000e+02                7              1        2021-03-23 18:30:00       9.870000e+02        1.033291                     1         2021-03-23 18:30:00
2     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps            15  1.983231e-01               15              1        2021-03-23 18:30:00       2.333000e-01        1.176363                     1         2021-03-23 18:30:00
3                         OSLinux-CPU_CPU_CPUidleutil            30  7.327371e+01               30              2        2021-03-23 18:31:00       7.332140e+01        1.000651                     1         2021-03-23 18:31:00
4                     OSLinux-CPU_CPU-2_SingleCpuidle            15  9.709545e+01               15              1        2021-03-23 18:32:00       9.738769e+01        1.003010                     1         2021-03-23 18:32:00
..                                                ...           ...           ...              ...            ...                        ...                ...             ...                   ...                         ...
43     OSLinux-OSLinux_FILESYSTEM_----home_FSCapacity            15  2.720228e+00               15              1        2021-03-23 18:59:00       2.720257e+00        1.000011                     1         2021-03-23 18:59:00
44        OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn            15  2.747013e+09               15              1        2021-03-23 18:59:00       2.747069e+09        1.000021                     1         2021-03-23 18:59:00
45       OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut            15  2.579300e+09               15              1        2021-03-23 18:59:00       2.579353e+09        1.000020                     1         2021-03-23 18:59:00
46  OSLinux-OSLinux_FILESYSTEM_----tomcat_FSInodeU...            15  2.938569e-01               15              1        2021-03-23 18:59:00       2.940060e-01        1.000507                     1         2021-03-23 18:59:00
47   OSLinux-OSLinux_FILESYSTEM_----tomcat_FSCapacity            15  9.692776e+01               15              1        2021-03-23 18:59:00       9.698374e+01        1.000577                     1         2021-03-23 18:59:00

[48 rows x 10 columns], 'MG02 has no KPI with longest_fault_length > 1.')

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[11]:

import pandas as pd
import numpy as np
import pytz

# reuse df_metric and tz if available
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

tz = pytz.timezone('Asia/Shanghai')

comp = 'MG02'
# Incident window (Asia/Shanghai) -> UTC epoch seconds
start_local = pd.Timestamp('2021-03-23 18:30:00', tz=tz)
end_local   = pd.Timestamp('2021-03-23 19:00:00', tz=tz)
start_epoch = int(start_local.tz_convert('UTC').timestamp())
end_epoch   = int(end_local.tz_convert('UTC').timestamp())

# 1) Filter MG02 series and select KPIs with 'network' or 'tcp' (case-insensitive)
mg02_all = df_metric[df_metric['cmdb_id'] == comp].copy()
mask_net = mg02_all['kpi_name'].str.contains('network|tcp', case=False, na=False)
mg02_net_all = mg02_all[mask_net].copy()

# If none, return empty results
if mg02_net_all.empty:
    final_table = pd.DataFrame(columns=[
        'kpi_name','total_points','global_p95','incident_points','anomaly_count',
        'earliest_anomaly_timestamp','max_anomaly_value','severity_ratio',
        'longest_fault_length','first_fault_start_timestamp'
    ])
    summary = (0, f"{comp} has no NETWORK/TCP KPIs in metric.csv.")
else:
    # 2) Compute global_p95 and total_points per KPI using full series (rule 9)
    mg02_net_global = (
        mg02_net_all
        .groupby('kpi_name', as_index=False)
        .agg(
            total_points=('value','count'),
            global_p95=('value', lambda s: float(s.quantile(0.95)))
        )
    )

    # 3) Incident-window rows for MG02 network/tcp KPIs
    mg02_net_inc = mg02_net_all[(mg02_net_all['timestamp'] >= start_epoch) & (mg02_net_all['timestamp'] <= end_epoch)].copy()
    if mg02_net_inc.empty:
        final_table = pd.DataFrame(columns=[
            'kpi_name','total_points','global_p95','incident_points','anomaly_count',
            'earliest_anomaly_timestamp','max_anomaly_value','severity_ratio',
            'longest_fault_length','first_fault_start_timestamp'
        ])
        summary = (0, f"{comp} has NETWORK/TCP KPIs but none have incident-window points.")
    else:
        # merge global thresholds
        mg02_net_inc = mg02_net_inc.merge(mg02_net_global, on='kpi_name', how='left')

        # incident_points per KPI
        incident_counts = mg02_net_inc.groupby('kpi_name', as_index=False).agg(incident_points=('value','count'))

        # anomaly points (value > global_p95)
        mg02_net_anom = mg02_net_inc[mg02_net_inc['value'] > mg02_net_inc['global_p95']].copy()

        if mg02_net_anom.empty:
            final_table = pd.DataFrame(columns=[
                'kpi_name','total_points','global_p95','incident_points','anomaly_count',
                'earliest_anomaly_timestamp','max_anomaly_value','severity_ratio',
                'longest_fault_length','first_fault_start_timestamp'
            ])
            summary = (0, f"{comp} has NETWORK/TCP incident points but no anomalies (> global_p95).")
        else:
            # aggregate basic anomaly stats per KPI
            anom_agg = (
                mg02_net_anom
                .groupby('kpi_name', as_index=False)
                .agg(
                    anomaly_count=('value','count'),
                    earliest_anomaly_timestamp_epoch=('timestamp','min'),
                    max_anomaly_value=('value','max')
                )
            )

            # compute fault segments per KPI for longest_fault_length and first fault start
            fault_rows = []
            grp = mg02_net_anom.sort_values(['kpi_name','timestamp']).groupby('kpi_name')
            for kpi, g in grp:
                times = g['timestamp'].values
                vals = g['value'].values
                # consecutive if diff == 60 seconds
                diffs = np.diff(times, prepend=times[0])
                seg_ids = (diffs != 60).cumsum()
                seg_df = pd.DataFrame({'ts': times, 'val': vals, 'seg': seg_ids})
                seg_summary = seg_df.groupby('seg').agg(
                    seg_start=('ts','min'),
                    seg_end=('ts','max'),
                    seg_len_points=('ts','count'),
                    seg_max_value=('val','max')
                ).reset_index(drop=True)
                longest_len = int(seg_summary['seg_len_points'].max())
                first_start_epoch = int(seg_summary['seg_start'].min())
                fault_rows.append({
                    'kpi_name': kpi,
                    'longest_fault_length': longest_len,
                    'first_fault_start_epoch': first_start_epoch
                })
            fault_df = pd.DataFrame(fault_rows)

            # combine aggregates
            combined = anom_agg.merge(incident_counts, on='kpi_name', how='left')
            combined = combined.merge(mg02_net_global, on='kpi_name', how='left')
            combined = combined.merge(fault_df, on='kpi_name', how='left')

            # compute severity_ratio (handle global_p95 == 0 -> inf if max>0)
            def compute_sev(row):
                gp = row['global_p95']
                mv = row['max_anomaly_value']
                if pd.isna(gp):
                    return np.nan
                if gp == 0:
                    return float('inf') if mv > 0 else np.nan
                return mv / gp
            combined['severity_ratio'] = combined.apply(compute_sev, axis=1)

            # convert epoch timestamps to Asia/Shanghai strings
            def epoch_to_local_str(v):
                if pd.isna(v):
                    return None
                return pd.to_datetime(int(v), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S')
            combined['earliest_anomaly_timestamp'] = combined['earliest_anomaly_timestamp_epoch'].apply(epoch_to_local_str)
            combined['first_fault_start_timestamp'] = combined['first_fault_start_epoch'].apply(epoch_to_local_str)

            # select final columns, sort, limit top 20
            final_table = combined[[
                'kpi_name','total_points','global_p95','incident_points',
                'anomaly_count','earliest_anomaly_timestamp','max_anomaly_value',
                'severity_ratio','longest_fault_length','first_fault_start_timestamp'
            ]].sort_values(by='earliest_anomaly_timestamp', ascending=True).head(20).reset_index(drop=True)

            # summary counts and long-fault check
            number_of_network_kpis_with_anomaly = int(final_table.shape[0])
            long_faults = final_table[final_table['longest_fault_length'] > 1]
            if long_faults.empty:
                long_fault_indicator = (False, None)
            else:
                # list KPI(s) with longest_fault_length >1 and their first start timestamps
                lf_list = list(long_faults[['kpi_name','first_fault_start_timestamp']].itertuples(index=False, name=None))
                long_fault_indicator = (True, lf_list)

            summary = (number_of_network_kpis_with_anomaly, long_fault_indicator)

# Output compact results
final_table, summary

```
Out[11]:
```
Summary for MG02 NETWORK/TCP KPIs (incident window 2021-03-23 18:30–19:00 UTC+8):

- Number of MG02 NETWORK/TCP KPIs that had anomaly points in the incident window: 7.

- Earliest anomalies (by timestamp):
  - 2021-03-23 18:34:00 — OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum (max=2911 vs p95=2819, severity≈1.033)
  - 2021-03-23 18:34:00 — OSLinux-OSLinux_NETWORK_ens160_NETBandwidthUtil (max≈0.0046 vs p95≈0.00454, severity≈1.013)
  - 2021-03-23 18:35:00 — OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT (max=1.0, global_p95=0 → severity = inf)
  - 2021-03-23 18:44:00 — OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT (max=1.0 vs p95=0.55, severity≈1.818)
  - Later anomalies at 18:50 and 18:59 for various ens160 metrics (NETKBTotalPerSec, NETPacketsIn, NETPacketsOut) with modest severity (~1.00–1.08).

- Notable items:
  - TCP-CLOSE-WAIT shows an infinite severity ratio because global_p95 == 0 while the spike value > 0 (one-minute positive spike).
  - TCP-FIN-WAIT has the largest finite relative spike (severity ≈ 1.82).
  - Most other network/TCP anomalies are modest (severity ≈ 1.00–1.08).

- Fault-lengths:
  - None of these NETWORK/TCP KPIs had a fault sub-series longer than 1 minute (longest_fault_length = 1 for all).
  - Therefore: no multi-minute continuous network/TCP faults detected for MG02 in this window.

Conclusion / recommended focus:
- Investigate the short spike in TCP-CLOSE-WAIT at 18:35 (inf severity due to p95==0) and the TCP-FIN-WAIT spike at 18:44, and check the tcp/connection metrics and the traffic around 18:34–18:44, since several connection/throughput metrics also show increases in that interval.

The original code execution output of IPython Kernel is also provided below for reference:

(                                          kpi_name  total_points    global_p95  incident_points  anomaly_count earliest_anomaly_timestamp  max_anomaly_value  severity_ratio  longest_fault_length first_fault_start_timestamp
0  OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum            21  2.819000e+03               21              1        2021-03-23 18:34:00       2.911000e+03        1.032636                     1         2021-03-23 18:34:00
1  OSLinux-OSLinux_NETWORK_ens160_NETBandwidthUtil             5  4.540000e-03                5              1        2021-03-23 18:34:00       4.600000e-03        1.013216                     1         2021-03-23 18:34:00
2   OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT            30  0.000000e+00               30              1        2021-03-23 18:35:00       1.000000e+00             inf                     1         2021-03-23 18:35:00
3     OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT            30  5.500000e-01               30              2        2021-03-23 18:44:00       1.000000e+00        1.818182                     1         2021-03-23 18:44:00
4  OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec            15  4.348954e+05               15              1        2021-03-23 18:50:00       4.700970e+05        1.080943                     1         2021-03-23 18:50:00
5      OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn            15  2.747013e+09               15              1        2021-03-23 18:59:00       2.747069e+09        1.000021                     1         2021-03-23 18:59:00
6     OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut            15  2.579300e+09               15              1        2021-03-23 18:59:00       2.579353e+09        1.000020                     1         2021-03-23 18:59:00, (7, (False, None)))```
```

In [ ]:
# In[12]:

import pandas as pd
import numpy as np
import pytz

# Reuse df_trace and trace_global if available; otherwise load / compute
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

tz = pytz.timezone('Asia/Shanghai')

# Ensure global p95 computed on full series (rule 9)
try:
    trace_global
except NameError:
    trace_global = (
        df_trace
        .groupby(['cmdb_id','trace_name'], as_index=False)
        .agg(
            total_points=('value','count'),
            global_p95=('value', lambda s: float(s.quantile(0.95)))
        )
    )

# Incident window epoch bounds (UTC+8 -> convert to UTC epoch)
filter_window_start_local = pd.Timestamp('2021-03-23 18:33:00', tz=tz)
filter_window_end_local   = pd.Timestamp('2021-03-23 18:35:00', tz=tz)
filter_start_epoch = int(filter_window_start_local.tz_convert('UTC').timestamp())
filter_end_epoch   = int(filter_window_end_local.tz_convert('UTC').timestamp())

incident_start_local = pd.Timestamp('2021-03-23 18:30:00', tz=tz)
incident_end_local   = pd.Timestamp('2021-03-23 19:00:00', tz=tz)
incident_start_epoch = int(incident_start_local.tz_convert('UTC').timestamp())
incident_end_epoch   = int(incident_end_local.tz_convert('UTC').timestamp())

# Filter trace rows to incident window and merge global_p95
df_trace_inc = df_trace[(df_trace['timestamp'] >= incident_start_epoch) & (df_trace['timestamp'] <= incident_end_epoch)].copy()
df_trace_inc = pd.merge(df_trace_inc, trace_global[['cmdb_id','trace_name','global_p95','total_points']],
                        on=['cmdb_id','trace_name'], how='left')

# incident_points per group (count of all incident-window rows)
incident_counts = df_trace_inc.groupby(['cmdb_id','trace_name'], as_index=False).agg(incident_points=('value','count'))

# anomaly rows in incident window (value > global_p95)
df_trace_anom = df_trace_inc[df_trace_inc['value'] > df_trace_inc['global_p95']].copy()

# Aggregate anomaly info per group
if not df_trace_anom.empty:
    anomaly_agg = (
        df_trace_anom
        .groupby(['cmdb_id','trace_name'], as_index=False)
        .agg(
            anomaly_count=('value','count'),
            earliest_anomaly_ts_epoch=('timestamp','min'),
            max_anomaly_value=('value','max')
        )
    )
    # merge with incident_counts and global p95
    merged = anomaly_agg.merge(incident_counts, on=['cmdb_id','trace_name'], how='left')
    merged = merged.merge(trace_global[['cmdb_id','trace_name','global_p95']], on=['cmdb_id','trace_name'], how='left')
    # severity ratio (handle global_p95==0)
    def compute_sev(row):
        gp = row['global_p95']
        mv = row['max_anomaly_value']
        if pd.isna(gp):
            return np.nan
        if gp == 0:
            return float('inf') if mv > 0 else np.nan
        return mv / gp
    merged['severity_ratio'] = merged.apply(compute_sev, axis=1)
else:
    merged = pd.DataFrame(columns=[
        'cmdb_id','trace_name','anomaly_count','earliest_anomaly_ts_epoch','max_anomaly_value','incident_points','global_p95','severity_ratio'
    ])

# Filter by earliest_anomaly_timestamp within [18:33,18:35] UTC+8 and trace_name contains 'duration' (case-insensitive)
if not merged.empty:
    # Filter by time window
    time_mask = (merged['earliest_anomaly_ts_epoch'] >= filter_start_epoch) & (merged['earliest_anomaly_ts_epoch'] <= filter_end_epoch)
    merged_time = merged[time_mask].copy()
    # Filter by trace_name substring 'duration' (case-insensitive) or specifically 'duration_p95'/'duration_mean'
    name_mask = merged_time['trace_name'].str.contains('duration', case=False, na=False)
    filtered = merged_time[name_mask].copy()
else:
    filtered = merged.copy()

# Prepare final display: convert epoch to UTC+8 string and select columns
def epoch_to_local_str(v):
    if pd.isna(v):
        return None
    return pd.to_datetime(int(v), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S')

if not filtered.empty:
    filtered['earliest_anomaly_timestamp'] = filtered['earliest_anomaly_ts_epoch'].apply(epoch_to_local_str)
    final_df = filtered[[
        'cmdb_id','trace_name','earliest_anomaly_timestamp','max_anomaly_value','global_p95','severity_ratio','incident_points'
    ]].sort_values(by='earliest_anomaly_timestamp', ascending=True).head(50).reset_index(drop=True)
    unique_cmdb_ids = set(final_df['cmdb_id'].unique())
else:
    final_df = filtered[[
        'cmdb_id','trace_name','earliest_anomaly_ts_epoch','max_anomaly_value','global_p95','severity_ratio','incident_points'
    ]].head(0)
    unique_cmdb_ids = set()

# Output compact results
final_df, unique_cmdb_ids

```
Out[12]:
```
Summary of trace-duration anomalies between 2021-03-23 18:33:00 and 18:35:00 (UTC+8):

Matches (sorted by earliest anomaly = all at 2021-03-23 18:35:00):

- MG01 — trace.from_dockerA1.duration_p95
  - earliest: 2021-03-23 18:35:00
  - max anomaly: 2.77325, global_p95: 1.39527, severity ≈ 1.988
  - incident_points: 30

- MG02 — trace.to_dockerA1.duration_mean
  - earliest: 2021-03-23 18:35:00
  - max anomaly: 0.33300, global_p95: 0.298611, severity ≈ 1.115
  - incident_points: 30

- MG02 — trace.to_dockerA1.duration_p95
  - earliest: 2021-03-23 18:35:00
  - max anomaly: 1.35690, global_p95: 1.309145, severity ≈ 1.036
  - incident_points: 30

- dockerA1 — trace.from_MG02.duration_mean
  - earliest: 2021-03-23 18:35:00
  - max anomaly: 0.33300, global_p95: 0.298611, severity ≈ 1.115
  - incident_points: 30

- dockerA1 — trace.from_MG02.duration_p95
  - earliest: 2021-03-23 18:35:00
  - max anomaly: 1.35690, global_p95: 1.309145, severity ≈ 1.036
  - incident_points: 30

- dockerA1 — trace.to_MG01.duration_p95
  - earliest: 2021-03-23 18:35:00
  - max anomaly: 2.77325, global_p95: 1.39527, severity ≈ 1.988
  - incident_points: 30

- dockerA2 — trace.self.duration_p95
  - earliest: 2021-03-23 18:35:00
  - max anomaly: 0.02000, global_p95: 0.01175, severity ≈ 1.702
  - incident_points: 31

Unique components in this filtered set: {MG01, MG02, dockerA1, dockerA2}.

Interpretation / guidance:
- All matching duration anomalies occur at the same minute (18:35:00 UTC+8), indicating a short, simultaneous latency spike across traces.
- Highest relative spikes (~2.0× p95) are on:
  - MG01 trace.from_dockerA1.duration_p95 and dockerA1 trace.to_MG01.duration_p95 (severity ≈ 1.99).
- dockerA2.self.duration_p95 also shows a notable relative increase (severity ≈ 1.70).
- MG02 ↔ dockerA1 traces show modest increases (severity ≈ 1.04–1.12).
- Recommended focus: investigate dockerA1 ↔ MG01 and dockerA2 around 18:35 for latency sources (network, service processing, or downstream contention).

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id                        trace_name earliest_anomaly_timestamp  max_anomaly_value  global_p95  severity_ratio  incident_points
0      MG01  trace.from_dockerA1.duration_p95        2021-03-23 18:35:00            2.77325    1.395270        1.987608               30
1      MG02   trace.to_dockerA1.duration_mean        2021-03-23 18:35:00            0.33300    0.298611        1.115164               30
2      MG02    trace.to_dockerA1.duration_p95        2021-03-23 18:35:00            1.35690    1.309145        1.036478               30
3  dockerA1     trace.from_MG02.duration_mean        2021-03-23 18:35:00            0.33300    0.298611        1.115164               30
4  dockerA1      trace.from_MG02.duration_p95        2021-03-23 18:35:00            1.35690    1.309145        1.036478               30
5  dockerA1        trace.to_MG01.duration_p95        2021-03-23 18:35:00            2.77325    1.395270        1.987608               30
6  dockerA2           trace.self.duration_p95        2021-03-23 18:35:00            0.02000    0.011750        1.702128               31, {'MG01', 'dockerA1', 'dockerA2', 'MG02'})```
```